## ICM Full Model Dashboard — User Manual (Step 1 to Step 6)

This dashboard runs an end-to-end workflow:

1. filter a sector reference MACC
2. scale it to firm MACCs
3. merge firm baseline + subsector parameters
4. run carbon market scenarios
5. export a full results workbook
6. generate plots + comparisons in a results dashboard

Everything is inside one Colab “wizard” with a **Go to:** dropdown.

---

# Before you start

### Files you must upload to Colab (`/content`)

Upload these 4 files (Step 1 will show ✅ when they exist):

* `reference_firm_macc_data.xlsx`
* `firm.xlsx`
* `firm_data_ccts.xlsx`
* `subsector_parameters.xlsx`

### What each file should contain

**A) `reference_firm_macc_data.xlsx`**

* Sheet: `reference_prod_and_emission`

  * columns: `sector`, `production`, `emission`
* For each sector, a sheet named:

  * `<sector>_sector_ref_macc` (lowercase)
  * columns: `intervention_id`, `intervention_name`, `abatement_cost`, `abatement_potential`

**B) `firm.xlsx`**

* columns: `registration_number`, `production`, `emission_intensity`, `sector`

**C) `firm_data_ccts.xlsx`**

* must contain at least:

  * `REGISTRATION_NUMBER`, `SECTOR`, `SUB_SECTOR`
  * `PROD_BASELINE`, `BASELINE_EI`, `TARGET_EI_2025_26`, `u_baseline`, `epsilon`

**D) `subsector_parameters.xlsx`**

* must contain:

  * `SECTOR`, `SUB_SECTOR`
  * plus subsector parameters used in the model (e.g., `epsilon` if you keep it there)

---

# Using the Wizard Navigation

Use the **Go to:** dropdown to switch steps:

* Step 1 — File status
* Step 2 — Filter reference
* Step 3 — Scale firm MACCs
* Step 4 — Build merged panel
* Step 5 — Scenario run + export
* Step 6 — Results dashboard

---

# Step 1 — Inputs & File Status

**Purpose:** confirm all files exist in Colab.

* Click **Refresh file status** any time.
* ✅ means the file exists.
* ❌ means missing or wrong filename.

You should see your 4 input files as ✅ before moving on.

---

# Step 2 — Filter Reference MACC (UI)

**Purpose:** choose which interventions from the reference MACC ladder are allowed to be used for scaling.

### How to use

1. Click **Initialize filter UI**
2. Use **Sector** dropdown to switch sectors
3. Use **Search** to filter interventions by ID/name
4. Select interventions in the list
5. Click **Apply to this sector** (this saves the selection)
6. Repeat for other sectors
7. Click **Write filtered workbook**

### Output

* `reference_firm_macc_data_filtered.xlsx`

### Advanced (custom add/delete)

Enable **Advanced (custom add/delete)** if you want to manually add new interventions:

* Choose ID (optional), name (required), cost, potential
* Units can be:

  * `abatement_potential (tCO2/yr)`
  * or `delta_ei (tCO2/unit)` (auto converted using sector reference production)
* You can delete custom options by selecting them and clicking delete.

---

# Step 3 — Scale Firm MACCs

**Purpose:** scale the filtered reference ladders into firm-specific MACCs using firm production and emission intensity.

### Controls

* **Allow negative EI**

  * If ON: the model allows the intensity to go below 0 (not physically realistic, but useful for stress tests)
  * If OFF: abatement stops once EI hits 0

* **Cleaner strategy**

  * `strict`: if a firm is already cleaner than the reference ladder coverage, it “skips” cheapest measures first (assumes they’ve already been done)
  * `full_remaining`: always applies the full ladder

* **Add additional_intervention if firm is dirtier**

  * If a firm’s EI is higher than the total ladder coverage, the code adds one synthetic measure to cover the gap

* **Additional cost offset**

  * Sets the synthetic measure cost = cheapest cost + offset
  * If offset is negative, synthetic measure can become cheaper than all others (interpret carefully)

### Run

Click **Run scaling → firm_macc_ccts.xlsx**

### Output

* `firm_macc_ccts.xlsx`
  (this is your scaled MACC workbook used by the carbon market model)

---

# Step 4 — Build merged firm panel

**Purpose:** create a single merged dataset combining:

* firm baseline/target data
* scaled MACC interventions
* subsector parameters

Click **Build merged panel → ICM_final_merged_firm_panel.xlsx**

### Output

* `ICM_final_merged_firm_panel.xlsx`
* Also stored in memory as `STATE["final_merged"]` for Step 5

---

# Step 5 — Scenario Table + Run

**Purpose:** run carbon market scenarios and export a full results workbook.

### Scenario table columns

* **[del]**: mark row to delete (then click Delete selected)

* **run**: include/exclude scenario from running

* **ID**: used for sheet naming in results (`SD_Curve_<ID>`, `Subsector_Detail_<ID>`)

* **delta** (`delta_ei_factor`): scales technology abatement potential

  * 1.0 = full potential
  * 0.3 = slow adoption (30% potential)

* **base** (`baseline_ei_factor`): scales baseline EI (autonomous technical progress)

  * 1.0 = no autonomous improvement
  * 0.98 = 2% autonomous improvement

* **floor**: minimum carbon price allowed (if floor toggle ON)

* **ceiling**: maximum carbon price allowed (if ceiling toggle ON)

* **filter neg cost**: drops options where `abatement_cost <= 0` or `delta_ei <= 0`

### Buttons

* **Load defaults**: loads standard A/B/C scenarios
* **Run: select all / select none**: quickly choose which scenarios run
* **Save config / Load config**: saves the scenario table to JSON for later reuse
* **Cancel run**: safe cancel (stops after the current operation finishes)

### Run controls (very important)

* **Eq step**
  Price search grid spacing for equilibrium search.
  Smaller = more accurate but slower.

* **Max cap**
  Maximum carbon price explored in the equilibrium search.
  If too large with small Eq step, it becomes extremely slow.

* **SD step**
  Resolution for supply-demand curve sheets (`SD_Curve_*`).
  This does not affect equilibrium; only affects curve detail.

### Progress indicators

* **Progress bar**: scenarios completed
* **Elapsed time**: runtime since Run started
* **ETA remaining**: appears after at least 2 scenarios complete
* **Status**: shows current scenario + stage (data_read, price_search, sd_curve, export, etc.)

### Output workbook (full)

After success, Step 5 writes:

`Carbon_Market_Scenarios_Results_v3.xlsx` containing:

* `Summary`
* `Firm_Level_All_Data`
* `SD_Curve_<scenario>`
* `Subsector_Detail_<scenario>`
* Pivot sheets:

  * `Subsector_Avg_Total_Cost`
  * `Subsector_Avg_Tech_Cost`
  * `Subsector_Avg_Carbon_Cost`
  * `Subsector_Cost_Ratio_Pct`
  * `Subsector_Emission_Red_Pct`
  * `Subsector_Intensity_Change`
  * `Subsector_Production_Change`

### Recommended settings

For fast iteration:

* Eq step = 0.1
* Max cap = 500 (or close to your max abatement cost)
* SD step = 2

For final runs:

* Eq step = 0.01
* Max cap = 5000–10000 (only if needed)
* SD step = 1 (or 0.5)

---

# Step 6 — Results Dashboard

**Purpose:** generate plots + scenario comparisons from the Step 5 results workbook.

### Start

Click **Launch Dashboard 🚀**

It will auto-load:

* `/content/Carbon_Market_Scenarios_Results_v3.xlsx`
* populate scenarios
* show KPI cards

### Main controls

* **Scenario**: choose scenario to plot
* **Level**: Auto/Subsector/Sector for impacts plots
* **DPI**: Draft vs HD quality
* **Plot selection**: choose which plot families to generate
* **Compare scenarios**: select scenario list + metric for comparison heatmap
* **View dropdown**: KPIs / Plots / Compare / Gallery

### Outputs

Plots are saved in:

* `dashboard_outputs/`

Buttons:

* **Generate Selected**: creates plots you selected
* **Generate HD + Zip**: produces high-resolution plots then zips and downloads
* **Show PNGs**: displays plot images
* **Export Tables**: exports KPI row + detail table + firm subset + compare pivot to Excel

### Plot families (what you get)

* **Impacts**

  * Carbon intensity change (%)
  * Unit cost change ratio (% of product price)
  * Output change (%)
  * Rich labels + color-based interpretation

* **Firm distributions**

  * del_u distribution by sector
  * output change distribution
  * N (buyer/seller) distribution
  * output sign count chart (negative/neutral/positive)

* **Market mechanics**

  * SD curve with price on Y-axis + equilibrium labels
  * imbalance vs price with price on Y-axis

* **Buyer/Seller**

  * stacked counts by sector
  * diverging volumes plot (demand vs supply)

* **Cost decomposition**

  * unit tech cost + unit carbon cost stacked

* **Scenario comparison heatmap**

  * compares chosen metric across chosen scenarios
  * uses Step 5 pivot sheets when available (fast)

---

# Typical workflow (recommended)

1. Step 1: confirm files ✅
2. Step 2: filter interventions → write filtered reference
3. Step 3: scale → produce firm_macc_ccts.xlsx
4. Step 4: merge → produce merged panel
5. Step 5: run baseline scenario first (B only) with fast settings
6. Step 5: run full scenario set with final settings
7. Step 6: generate plots and download zip for PPT/report

---






# installations

In [ ]:
!pip -q install "ipywidgets==7.7.1" openpyxl xlsxwriter
!pip -q install python-calamine

# step 1

In [ ]:
# ======================================================
# IMPORTS & SETUP
# ======================================================
from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import pandas as pd
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
import re, os, json, warnings, time
warnings.filterwarnings("ignore")

# ----------------------------
# CONFIGURATION
# ----------------------------
FILES_REQUIRED = [
    "reference_facility_macc_data.xlsx",
    "facility.xlsx",
    "facility_data_ccts.xlsx",
    "subsector_parameters.xlsx"
]

PATHS = {
    "ref_in": "reference_facility_macc_data.xlsx",
    "ref_filtered": "reference_facility_macc_data_filtered.xlsx",
    "scaled_book": "scaled_INTENSITY_macc_1.xlsx",
    "facility_macc_ccts": "facility_macc_ccts.xlsx",
    "merged_panel": "ICM_final_merged_facility_panel.xlsx",
    "scenario_results": "Carbon_Market_Scenarios_Results_v3.xlsx",
}

STATE = {
    "final_merged": None,             # set in Step 4
    "filter_ui_ready": False,         # set in Step 2
    "filter_selected": None,          # dict: selected_ids_by_sector
    "filter_sector_tables": None,     # sector_tables (mutable, includes custom)
    "filter_df_index": None,          # df_index
    "filter_sector_keys": None,       # list of sectors
    "filter_ref_prod_by_sector": None # dict: sector -> prod
}

# ----------------------------
# STYLING
# ----------------------------
display(widgets.HTML("""
<style>
.widget-select-multiple select, .widget-select select, select {
    font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, monospace !important;
    white-space: pre !important;
}
.widget-button button { white-space: nowrap !important; }
.status-table { width: 100%; border-collapse: collapse; font-family: sans-serif; margin-top: 10px; }
.status-table th { text-align: left; border-bottom: 2px solid #ddd; padding: 8px; background-color: #f8f9fa; }
.status-table td { border-bottom: 1px solid #eee; padding: 8px; }
.status-pass { color: #28a745; font-weight: bold; }
.status-fail { color: #dc3545; font-weight: bold; }
.type-badge { font-size: 0.75em; padding: 2px 6px; border-radius: 4px; background: #e9ecef; color: #495057; border: 1px solid #ced4da; }
</style>
"""))

# ----------------------------
# HELPERS
# ----------------------------
def _norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    return df

def _require(df: pd.DataFrame, cols: list, where: str):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing} in {where}. Found: {list(df.columns)}")

def _compose_sector_sheet_name(sector: str) -> str:
    return f"{sector.strip().lower()}_sector_ref_macc"

INVALID_SHEET_CHARS = r"[\[\]\:\*\?\/\\]"
def safe_excel_sheet_name(raw: str, used: set, fallback="facility") -> str:
    name = (raw or "").strip()
    name = re.sub(INVALID_SHEET_CHARS, "_", name)
    name = name[:31].strip() or fallback
    base = name
    k = 1
    # Note: 'used' set must be managed by the caller
    while name in used:
        suffix = f"_{k}"
        name = (base[:31-len(suffix)] + suffix) if len(base) + len(suffix) > 31 else (base + suffix)
        k += 1
    used.add(name)
    return name

# ======================================================
# STEP 1 — File status (Polished)
# ======================================================
file_status_widget = widgets.HTML()

def get_file_status_html():
    # logical separation of Inputs vs Outputs for the report
    inputs = {f: "Input" for f in FILES_REQUIRED}
    # Outputs are in PATHS, but we exclude any that are already listed as Inputs
    outputs = {v: "Output" for k, v in PATHS.items() if v not in inputs}

    all_files = {**inputs, **outputs}

    rows = []
    for filename, ftype in all_files.items():
        exists = Path(filename).exists()
        icon = "✅" if exists else "❌"
        status_class = "status-pass" if exists else "status-fail"
        status_text = "Ready" if exists else "Missing"

        rows.append(f"""
            <tr>
                <td style="text-align:center;">{icon}</td>
                <td style="font-family:monospace;">{filename}</td>
                <td><span class="type-badge">{ftype}</span></td>
                <td class="{status_class}">{status_text}</td>
            </tr>
        """)

    return f"""
    <table class="status-table">
        <thead><tr><th style="width:40px"></th><th>Filename</th><th>Type</th><th>Status</th></tr></thead>
        <tbody>{''.join(rows)}</tbody>
    </table>
    """

def refresh_file_status(_=None):
    file_status_widget.value = get_file_status_html()

btn_refresh_files = widgets.Button(
    description="Refresh Status",
    button_style="info",
    icon="refresh",
    layout=widgets.Layout(width="150px")
)
btn_refresh_files.on_click(refresh_file_status)

# Run once on load
refresh_file_status()

step1_box = widgets.VBox([
    widgets.HTML("<h3>📂 Step 1: File Initialization</h3>"),
    widgets.HTML(
        "<p style='color:#666; margin-bottom:15px;'>"
        "Please ensure the following <b>Input</b> files are uploaded to the file system before proceeding."
        "</p>"
    ),
    btn_refresh_files,
    file_status_widget
], layout=widgets.Layout(padding="15px", border="1px solid #e0e0e0", margin="10px 0", width="98%"))

HTML(value='\n<style>\n.widget-select-multiple select, .widget-select select, select {\n    font-family: ui-mo…

# step 2

v1

In [ ]:
# # ======================================================
# # STEP 2 — Filter reference MACC (Polished)
# # ======================================================

# # Container for the step
# step2_container = widgets.VBox()
# step2_msg_out = widgets.Output()

# # ----------------------------
# # LOGIC: Load & Filter
# # ----------------------------
# def load_reference_workbook(ref_xlsx: str):
#     path = Path(ref_xlsx)
#     if not path.exists():
#         raise FileNotFoundError(f"Reference workbook not found: {ref_xlsx}")

#     # Load Index
#     df_index = pd.read_excel(path, sheet_name="reference_prod_and_emission")
#     df_index = _norm_cols(df_index)
#     _require(df_index, ["sector", "production", "emission"], "reference_prod_and_emission")

#     df_index = df_index.dropna(subset=["sector"]).copy()
#     df_index["sector"] = df_index["sector"].astype(str).str.strip()
#     df_index = df_index[df_index["sector"] != ""]

#     # Load Sectors
#     xls = pd.ExcelFile(path)
#     all_sheets = set(xls.sheet_names)

#     sector_tables = {}
#     for _, r in df_index.iterrows():
#         sector = str(r["sector"]).strip()
#         sheet = _compose_sector_sheet_name(sector)

#         # Graceful skip or error? (Sticking to your strict logic)
#         if sheet not in all_sheets:
#             raise ValueError(f"Missing sheet '{sheet}' for sector '{sector}'")

#         df = pd.read_excel(path, sheet_name=sheet)
#         df = _norm_cols(df)

#         # Normalize column names
#         if "abatement potential" in df.columns and "abatement_potential" not in df.columns:
#             df = df.rename(columns={"abatement potential": "abatement_potential"})

#         col_map = {
#             "intervention_id": ["intervention_id", "id"],
#             "intervention_name": ["intervention_name", "measure", "name"],
#             "abatement_cost": ["abatement_cost", "cost", "cost_inr_per_tco2"],
#             "abatement_potential": ["abatement_potential", "width_tco2_per_year", "abatement_width"],
#         }

#         mapped = {}
#         for std, cands in col_map.items():
#             for c in cands:
#                 if c in df.columns:
#                     mapped[std] = c
#                     break
#             if std not in mapped:
#                 raise ValueError(f"Sheet '{sheet}' missing column for '{std}'")

#         df = df.rename(columns=mapped)

#         # Validate Data
#         _require(df, ["intervention_id","intervention_name","abatement_cost","abatement_potential"], f"sheet '{sheet}'")
#         if df["abatement_cost"].isna().any() or df["abatement_potential"].isna().any():
#             raise ValueError(f"Nulls found in '{sheet}'. Please fill all costs/potentials.")

#         # Sort for UI
#         df = df.sort_values("abatement_cost").reset_index(drop=True)
#         sector_tables[sector.strip().lower()] = {"sheet": sheet, "df": df}

#     return df_index, sector_tables

# def write_filtered_reference(df_index, sector_tables_filtered, out_xlsx: str):
#     with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as w:
#         df_index.to_excel(w, sheet_name="reference_prod_and_emission", index=False)
#         for sect, payload in sector_tables_filtered.items():
#             payload["df"].to_excel(w, sheet_name=payload["sheet"], index=False)

# # ----------------------------
# # UI CONSTRUCTION
# # ----------------------------
# def init_filter_ui(_=None):
#     step2_container.children = [widgets.HTML("<i>Loading reference workbook... this may take a moment.</i>")]

#     try:
#         # 1. Load Data
#         df_index, sector_tables = load_reference_workbook(PATHS["ref_in"])
#         sector_keys = sorted(sector_tables.keys())

#         if not sector_keys:
#             step2_container.children = [widgets.HTML("<b style='color:red'>Error: No sectors found in index file.</b>")]
#             return

#         # 2. Init State
#         ref_prod_by_sector = {str(r["sector"]).strip().lower(): float(r["production"]) for _, r in df_index.iterrows()}
#         selected_ids_by_sector = {s: set(sector_tables[s]["df"]["intervention_id"].astype(str).tolist()) for s in sector_keys}
#         custom_ids_by_sector = {s: set() for s in sector_keys}

#         STATE.update({
#             "filter_ui_ready": True,
#             "filter_df_index": df_index,
#             "filter_sector_tables": sector_tables,
#             "filter_sector_keys": sector_keys,
#             "filter_selected": selected_ids_by_sector,
#             "filter_ref_prod_by_sector": ref_prod_by_sector,
#             "filter_custom": custom_ids_by_sector
#         })

#         # 3. Build Widgets
#         # Use continuous_update=False to prevent server lag on typing
#         search = widgets.Text(placeholder="Type & hit Enter...", description="Search:", continuous_update=False)
#         sector_dd = widgets.Dropdown(options=sector_keys, description="Sector:")

#         sel = widgets.SelectMultiple(options=[], value=(), rows=15, layout=widgets.Layout(width="98%"))
#         header = widgets.HTML(layout=widgets.Layout(margin="0 0 5px 0"))
#         status_lbl = widgets.HTML(layout=widgets.Layout(margin="5px 0"))
#         preview_out = widgets.Output(layout=widgets.Layout(border="1px solid #eee", padding="5px", max_height="300px", overflow="scroll"))

#         # Buttons
#         btn_all = widgets.Button(description="Select Visible", icon="check-circle-o")
#         btn_none = widgets.Button(description="Clear Visible", icon="circle-o", button_style="warning")
#         btn_apply = widgets.Button(description="Update View", icon="refresh")
#         btn_write = widgets.Button(description="Save Filtered Workbook", button_style="success", icon="save", layout=widgets.Layout(width="250px"))

#         # Width constants for the "Fake Table" in SelectMultiple
#         NAME_W, MAX_ID_W = 45, 15

#         def _sector_widths(sect):
#             df = sector_tables[sect]["df"]
#             id_w = int(df["intervention_id"].astype(str).map(len).max()) if len(df) else 10
#             return max(10, min(MAX_ID_W, id_w)), NAME_W

#         def _make_options(sect, q=""):
#             df = sector_tables[sect]["df"]
#             q = (q or "").strip().lower()
#             id_w, name_w = _sector_widths(sect)

#             opts = []
#             for _, r in df.iterrows():
#                 # Format: ID | Name | Cost | Potential
#                 # Using fixed width fonts requires non-breaking spaces sometimes, but standard spaces work in <pre>
#                 label_str = (
#                     f"{str(r['intervention_id'])[:id_w]:<{id_w}}  "
#                     f"{str(r['intervention_name'])[:name_w]:<{name_w}}  "
#                     f"{float(r['abatement_cost']):>10.1f}  "
#                     f"{float(r['abatement_potential']):>12,.0f}"
#                 )
#                 if (not q) or (q in label_str.lower()):
#                     opts.append((label_str, str(r["intervention_id"])))
#             return opts

#         def render_ui_state():
#             sect = sector_dd.value

#             # 1. Update List
#             opts = _make_options(sect, search.value)
#             sel.options = opts
#             visible_vals = {v for _, v in opts}

#             # Preserve selection state
#             current_selection = selected_ids_by_sector[sect]
#             sel.value = tuple([v for v in current_selection if v in visible_vals])

#             # 2. Update Headers/Status
#             id_w, name_w = _sector_widths(sect)
#             header.value = f"<pre class='macc-pre' style='font-weight:bold; border-bottom:1px solid #ccc; padding-bottom:4px; margin-bottom:0;'>{'ID':<{id_w}}  {'Name':<{name_w}}  {'Cost (INR)':>10}  {'Potential':>12}</pre>"

#             total_items = len(sector_tables[sect]["df"])
#             sel_count = len(current_selection)
#             cust_count = len(custom_ids_by_sector[sect])
#             status_lbl.value = f"Sector: <b>{sect.upper()}</b> | Selected: <b>{sel_count}/{total_items}</b> | Custom Added: <b>{cust_count}</b>"

#             # 3. Render Preview Table (Only top 100 to save rendering time)
#             with preview_out:
#                 clear_output()
#                 df = sector_tables[sect]["df"].copy()
#                 df = df[df["intervention_id"].isin(current_selection)].sort_values("abatement_cost")

#                 if df.empty:
#                     display(widgets.HTML("<em>No interventions selected.</em>"))
#                 else:
#                     # Clean up for display
#                     df_disp = df[["intervention_id", "intervention_name", "abatement_cost", "abatement_potential"]].copy()
#                     df_disp.columns = ["ID", "Name", "Cost", "Potential"]
#                     # Style the table
#                     display(widgets.HTML(df_disp.to_html(index=False, classes="status-table", max_rows=20)))

#         # Callbacks
#         def on_selection_change(change):
#             # When user manually clicks rows in the list
#             if change['name'] == 'value':
#                 sect = sector_dd.value
#                 visible_vals = {v for _, v in sel.options}
#                 # Everything visible that IS NOT in the new value has been deselected
#                 # Everything visible that IS in the new value is selected
#                 # We must be careful not to wipe out hidden selections (from search filtering)
#                 hidden_selections = selected_ids_by_sector[sect] - visible_vals
#                 selected_ids_by_sector[sect] = hidden_selections | set(change['new'])

#                 # Refresh just the preview, not the whole list (prevent flicker)
#                 # But for simplicity in this architecture, we might re-render stats
#                 total_items = len(sector_tables[sect]["df"])
#                 sel_count = len(selected_ids_by_sector[sect])
#                 cust_count = len(custom_ids_by_sector[sect])
#                 status_lbl.value = f"Sector: <b>{sect.upper()}</b> | Selected: <b>{sel_count}/{total_items}</b> | Custom Added: <b>{cust_count}</b>"

#                 # Debounce preview update? For now direct:
#                 with preview_out:
#                     clear_output()
#                     df = sector_tables[sect]["df"]
#                     df = df[df["intervention_id"].isin(selected_ids_by_sector[sect])].sort_values("abatement_cost")
#                     df_disp = df[["intervention_id", "intervention_name", "abatement_cost", "abatement_potential"]].copy()
#                     df_disp.columns = ["ID", "Name", "Cost", "Potential"]
#                     display(widgets.HTML(df_disp.to_html(index=False, classes="status-table", max_rows=20)))

#         sel.observe(on_selection_change, names='value')

#         def on_bulk_select(b):
#             sect = sector_dd.value
#             visible_vals = {v for _, v in sel.options}
#             current = selected_ids_by_sector[sect]
#             if b.description == "Select Visible":
#                 selected_ids_by_sector[sect] = current | visible_vals
#             else:
#                 selected_ids_by_sector[sect] = current - visible_vals
#             render_ui_state()

#         btn_all.on_click(on_bulk_select)
#         btn_none.on_click(on_bulk_select)

#         sector_dd.observe(lambda _: render_ui_state(), names='value')
#         search.observe(lambda _: render_ui_state(), names='value') # Optimized by continuous_update=False
#         btn_apply.on_click(lambda _: render_ui_state())

#         def on_save(_):
#             filtered = {}
#             for sect in sector_keys:
#                 df = sector_tables[sect]["df"].copy()
#                 # Keep only selected
#                 keep = selected_ids_by_sector[sect]
#                 df2 = df[df["intervention_id"].astype(str).isin(keep)].copy()
#                 df2 = df2.sort_values("abatement_cost").reset_index(drop=True)
#                 filtered[sect] = {"sheet": sector_tables[sect]["sheet"], "df": df2}

#             write_filtered_reference(df_index, filtered, PATHS["ref_filtered"])

#             with step2_msg_out:
#                 clear_output()
#                 display(widgets.HTML(f"<div class='status-pass'>✅ Saved filtered workbook to: {PATHS['ref_filtered']}</div>"))
#                 time.sleep(3)
#                 clear_output()

#         btn_write.on_click(on_save)

#         # 4. Advanced Panel (Custom Add)
#         # -----------------------------
#         adv_inputs = {
#             "id": widgets.Text(placeholder="Auto-ID if empty", description="ID:", layout=widgets.Layout(width="200px")),
#             "name": widgets.Text(placeholder="New Measure Name", description="Name:", layout=widgets.Layout(width="300px")),
#             "cost": widgets.FloatText(value=0, description="Cost:", layout=widgets.Layout(width="150px")),
#             "pot": widgets.FloatText(value=0, description="Pot:", layout=widgets.Layout(width="150px")),
#             "unit": widgets.Dropdown(options=[("tCO2/yr", "tco2yr"), ("Intensity", "delta_ei")], value="tco2yr", layout=widgets.Layout(width="100px"))
#         }
#         btn_add_cust = widgets.Button(description="Add", button_style="primary", layout=widgets.Layout(width="80px"))

#         def on_add_custom(_):
#             sect = sector_dd.value
#             nm = adv_inputs["name"].value.strip()
#             if not nm: return

#             # ID Gen
#             proposed = adv_inputs["id"].value.strip() or "CUST"
#             existing = set(sector_tables[sect]["df"]["intervention_id"].astype(str))
#             iid = proposed
#             k=1
#             while iid in existing:
#                 iid = f"{proposed}_{k}"
#                 k+=1

#             # Calc
#             cost = adv_inputs["cost"].value
#             pot_raw = adv_inputs["pot"].value
#             pot = pot_raw * ref_prod_by_sector[sect] if adv_inputs["unit"].value == "delta_ei" else pot_raw

#             # Update DF
#             new_row = {"intervention_id": iid, "intervention_name": nm, "abatement_cost": cost, "abatement_potential": pot}
#             df = sector_tables[sect]["df"]
#             sector_tables[sect]["df"] = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True).sort_values("abatement_cost")

#             custom_ids_by_sector[sect].add(str(iid))
#             selected_ids_by_sector[sect].add(str(iid))

#             # Reset & Refresh
#             adv_inputs["name"].value = ""
#             adv_inputs["id"].value = ""
#             render_ui_state()

#         btn_add_cust.on_click(on_add_custom)

#         adv_box = widgets.HBox(
#             [adv_inputs["id"], adv_inputs["name"], adv_inputs["cost"], adv_inputs["pot"], adv_inputs["unit"], btn_add_cust],
#             layout=widgets.Layout(width="98%", flex_flow="row wrap", border="1px dashed #ccc", padding="8px")
#         )
#         adv_accord = widgets.Accordion(children=[adv_box], selected_index=None)
#         adv_accord.set_title(0, "Advanced: Add Custom Intervention")

#         # 5. Final Assembly
#         step2_container.children = [
#             widgets.HBox([sector_dd, search, btn_write], layout=widgets.Layout(justify_content="space-between", margin="0 0 10px 0")),
#             widgets.HBox([widgets.VBox([header, sel, widgets.HBox([btn_all, btn_none])], layout=widgets.Layout(width="60%")),
#                           widgets.VBox([widgets.Label("Selected Preview:"), preview_out], layout=widgets.Layout(width="40%", padding="0 0 0 10px"))]),
#             status_lbl,
#             adv_accord,
#             step2_msg_out
#         ]

#         # Initial Render
#         render_ui_state()

#     except Exception as e:
#         step2_container.children = [widgets.HTML(f"<b style='color:red'>Error initializing Step 2: {str(e)}</b>")]
#         # raise e # Uncomment for debug

# btn_init = widgets.Button(description="Load & Initialize Filter", button_style="info", icon="filter", layout=widgets.Layout(width="200px"))
# btn_init.on_click(init_filter_ui)

# step2_box = widgets.VBox([
#     widgets.HTML("<h3>🛠️ Step 2: Filter Reference MACC</h3>"),
#     widgets.HTML("Select interventions to include in the simulation."),
#     btn_init,
#     step2_container
# ], layout=widgets.Layout(padding="10px", border="1px solid #e0e0e0"))

# v2

In [ ]:
# ======================================================
# STEP 2 — Filter & Edit Reference MACC (Final with ID Edit)
# ======================================================
import matplotlib.pyplot as plt

# Container for the step
step2_container = widgets.VBox()
step2_msg_out = widgets.Output()

# Force Monospace Alignment CSS
display(widgets.HTML("""
<style>
.macc-list select {
    font-family: "Courier New", Courier, monospace !important;
    white-space: pre !important;
    font-size: 13px !important;
}
.macc-header {
    font-family: "Courier New", Courier, monospace !important;
    white-space: pre !important;
    font-size: 13px !important;
    font-weight: bold;
    border-bottom: 2px solid #ccc;
    background-color: #f9f9f9;
    padding: 5px 0;
    margin-bottom: 0;
}
</style>
"""))

# ----------------------------
# LOGIC: Load & Filter
# ----------------------------
def load_reference_workbook(ref_xlsx: str):
    path = Path(ref_xlsx)
    if not path.exists():
        raise FileNotFoundError(f"Reference workbook not found: {ref_xlsx}")

    # Load Index
    df_index = pd.read_excel(path, sheet_name="reference_prod_and_emission")
    df_index = _norm_cols(df_index)
    _require(df_index, ["sector", "production", "emission"], "reference_prod_and_emission")

    df_index = df_index.dropna(subset=["sector"]).copy()
    df_index["sector"] = df_index["sector"].astype(str).str.strip()
    df_index = df_index[df_index["sector"] != ""]

    # Load Sectors
    xls = pd.ExcelFile(path)
    all_sheets = set(xls.sheet_names)

    sector_tables = {}
    for _, r in df_index.iterrows():
        sector = str(r["sector"]).strip()
        sheet = _compose_sector_sheet_name(sector)

        if sheet not in all_sheets:
            raise ValueError(f"Missing sheet '{sheet}' for sector '{sector}'")

        df = pd.read_excel(path, sheet_name=sheet)
        df = _norm_cols(df)

        if "abatement potential" in df.columns and "abatement_potential" not in df.columns:
            df = df.rename(columns={"abatement potential": "abatement_potential"})

        col_map = {
            "intervention_id": ["intervention_id", "id"],
            "intervention_name": ["intervention_name", "measure", "name"],
            "abatement_cost": ["abatement_cost", "cost", "cost_inr_per_tco2"],
            "abatement_potential": ["abatement_potential", "width_tco2_per_year", "abatement_width"],
        }
        mapped = {}
        for std, cands in col_map.items():
            for c in cands:
                if c in df.columns:
                    mapped[std] = c
                    break
            if std not in mapped:
                raise ValueError(f"Sheet '{sheet}' missing column for '{std}'")

        df = df.rename(columns=mapped)
        _require(df, ["intervention_id","intervention_name","abatement_cost","abatement_potential"], f"sheet '{sheet}'")

        # Ensure numeric
        df["abatement_cost"] = pd.to_numeric(df["abatement_cost"], errors="coerce").fillna(0)
        df["abatement_potential"] = pd.to_numeric(df["abatement_potential"], errors="coerce").fillna(0)

        df = df.sort_values("abatement_cost").reset_index(drop=True)
        sector_tables[sector.strip().lower()] = {"sheet": sheet, "df": df}

    return df_index, sector_tables

def write_filtered_reference(df_index, sector_tables_filtered, out_xlsx: str):
    with pd.ExcelWriter(out_xlsx, engine="xlsxwriter") as w:
        df_index.to_excel(w, sheet_name="reference_prod_and_emission", index=False)
        for sect, payload in sector_tables_filtered.items():
            payload["df"].to_excel(w, sheet_name=payload["sheet"], index=False)

# ----------------------------
# UI CONSTRUCTION
# ----------------------------
def init_filter_ui(_=None):
    step2_container.children = [widgets.HTML("<i>Loading reference workbook...</i>")]

    try:
        # 1. Load Data
        df_index, sector_tables = load_reference_workbook(PATHS["ref_in"])
        sector_keys = sorted(sector_tables.keys())

        if not sector_keys:
            step2_container.children = [widgets.HTML("<b style='color:red'>Error: No sectors found.</b>")]
            return

        # 2. Init State
        ref_prod_by_sector = {str(r["sector"]).strip().lower(): float(r["production"]) for _, r in df_index.iterrows()}
        selected_ids_by_sector = {s: set(sector_tables[s]["df"]["intervention_id"].astype(str).tolist()) for s in sector_keys}
        custom_ids_by_sector = {s: set() for s in sector_keys}

        STATE.update({
            "filter_ui_ready": True,
            "filter_sector_tables": sector_tables,
            "filter_sector_keys": sector_keys,
            "filter_selected": selected_ids_by_sector,
            "filter_custom": custom_ids_by_sector
        })

        # 3. Widgets
        search = widgets.Text(placeholder="Search...", description="Filter:", continuous_update=False, layout=widgets.Layout(width="250px"))
        sector_dd = widgets.Dropdown(options=sector_keys, description="Sector:", layout=widgets.Layout(width="300px"))

        # Add 'macc-list' class for CSS targeting
        sel = widgets.SelectMultiple(options=[], value=(), rows=16, layout=widgets.Layout(width="98%"))
        sel.add_class("macc-list")

        header = widgets.HTML(layout=widgets.Layout(margin="0", width="98%"))
        header.add_class("macc-header")

        # Visuals
        plot_out = widgets.Output(layout=widgets.Layout(height="300px", border="1px solid #ddd", margin="10px 0"))

        # Actions
        btn_all = widgets.Button(description="Select All", icon="check", layout=widgets.Layout(width="120px"))
        btn_none = widgets.Button(description="Clear", icon="times", button_style="warning", layout=widgets.Layout(width="100px"))
        btn_write = widgets.Button(description="Save Filtered File", button_style="success", icon="save", layout=widgets.Layout(width="180px"))

        # Cost Filter
        cost_slider = widgets.FloatSlider(value=0, min=-5000, max=15000, step=10, description="Cost Limit:", readout_format=".0f", layout=widgets.Layout(width="400px"))

        btn_filter_cost = widgets.Button(
            description="Deselect < Limit",
            icon="minus-circle",
            button_style="warning",
            layout=widgets.Layout(width="160px"),
            tooltip="Remove all interventions CHEAPER than this cost limit"
        )

        # Edit/Add Panel (Widened Potential)
        edit_id = widgets.Text(description="ID:", disabled=False, layout=widgets.Layout(width="200px"))
        edit_name = widgets.Text(description="Name:", layout=widgets.Layout(width="350px"))
        edit_cost = widgets.FloatText(description="Cost:", layout=widgets.Layout(width="150px"))
        edit_pot = widgets.FloatText(description="Potential:", layout=widgets.Layout(width="300px"))
        status_msg = widgets.Label(value="")

        btn_update = widgets.Button(description="Update Selected", button_style="primary", disabled=True, icon="pencil")
        btn_add_new = widgets.Button(description="Add New", button_style="info", icon="plus")
        btn_delete = widgets.Button(description="Delete", button_style="danger", icon="trash")

        # --- LOGIC ---
        NAME_W = 55
        MAX_ID_W = 15

        def _make_options(sect, q=""):
            df = sector_tables[sect]["df"]
            q = q.strip().lower()
            id_w = int(df["intervention_id"].astype(str).map(len).max()) if len(df) else 10
            id_w = max(10, min(MAX_ID_W, id_w))

            opts = []
            for _, r in df.iterrows():
                # Precise formatting to match header
                lbl = (f"{str(r['intervention_id'])[:id_w]:<{id_w}}  "
                       f"{str(r['intervention_name'])[:NAME_W]:<{NAME_W}}  "
                       f"{float(r['abatement_cost']):>12.1f}  "
                       f"{float(r['abatement_potential']):>18,.0f}")
                if not q or q in lbl.lower():
                    opts.append((lbl, str(r["intervention_id"])))
            return opts, id_w

        def render_plot():
            sect = sector_dd.value
            with plot_out:
                clear_output(wait=True)
                df = sector_tables[sect]["df"].copy()
                # Filter to selected only
                sel_ids = selected_ids_by_sector[sect]
                df = df[df["intervention_id"].astype(str).isin(sel_ids)].sort_values("abatement_cost")

                if df.empty:
                    print("No interventions selected.")
                    return

                # MACC Plot
                costs = df["abatement_cost"].values
                widths = df["abatement_potential"].values
                cum_w = np.cumsum(widths)
                x = np.concatenate(([0], cum_w))

                plt.figure(figsize=(9, 3.5))
                plt.bar(x[:-1], costs, width=widths, align='edge', color='skyblue', edgecolor='black', alpha=0.7)
                plt.axhline(0, color='black', linewidth=1)
                plt.xlabel("Abatement Potential (tCO2)")
                plt.ylabel("Cost ($/tCO2)")
                plt.title(f"MACC Preview: {sect.title()}")
                plt.grid(True, alpha=0.3)
                plt.tight_layout()
                plt.show()

        def refresh_ui():
            sect = sector_dd.value
            opts, id_w = _make_options(sect, search.value)
            sel.options = opts

            # Restore selection
            current = selected_ids_by_sector[sect]
            visible = {v for _, v in opts}
            sel.value = tuple([v for v in current if v in visible])

            # Header with matching format
            header.value = (f"<div class='macc-header'>"
                            f"{'ID':<{id_w}}  {'Name':<{NAME_W}}  {'Cost':>12}  {'Potential':>18}</div>")
            render_plot()

        # --- EVENT HANDLERS ---

        def on_select_change(change):
            if change['name'] != 'value':
                return

            sect = sector_dd.value
            visible_vals = {v for _, v in sel.options}
            hidden_selections = selected_ids_by_sector[sect] - visible_vals
            selected_ids_by_sector[sect] = hidden_selections | set(change['new'])

            # Populate Fields
            if len(change['new']) == 1:
                sid = change['new'][0]
                df = sector_tables[sect]["df"]
                row = df[df["intervention_id"].astype(str) == sid]
                if not row.empty:
                    r = row.iloc[0]
                    edit_id.value = str(r["intervention_id"])
                    edit_name.value = str(r["intervention_name"])
                    edit_cost.value = float(r["abatement_cost"])
                    edit_pot.value = float(r["abatement_potential"])
                    btn_update.disabled = False
                    status_msg.value = f"Editing: {sid}"
            else:
                btn_update.disabled = True
                edit_id.value = ""
                edit_name.value = ""
                edit_cost.value = 0.0
                edit_pot.value = 0.0
                status_msg.value = ""

            render_plot()

        def on_update(_):
            sect = sector_dd.value

            # Get original ID from selection (if 1 selected)
            if len(sel.value) != 1: return
            original_id = sel.value[0]
            new_id = edit_id.value.strip()

            df = sector_tables[sect]["df"]

            # If renaming, check for duplicate
            if new_id != original_id:
                if new_id in set(df["intervention_id"].astype(str)):
                    status_msg.value = "⚠️ Error: New ID already exists!"
                    return

            # Locate row
            mask = df["intervention_id"].astype(str) == original_id
            if mask.any():
                idx = df[mask].index[0]
                # Update Values
                sector_tables[sect]["df"].at[idx, "intervention_id"] = new_id
                sector_tables[sect]["df"].at[idx, "intervention_name"] = edit_name.value
                sector_tables[sect]["df"].at[idx, "abatement_cost"] = edit_cost.value
                sector_tables[sect]["df"].at[idx, "abatement_potential"] = edit_pot.value

                # Update Selection Set: Remove old ID, add new ID
                selected_ids_by_sector[sect].remove(original_id)
                selected_ids_by_sector[sect].add(new_id)

                # Update Custom Set if applicable
                if original_id in custom_ids_by_sector[sect]:
                    custom_ids_by_sector[sect].remove(original_id)
                    custom_ids_by_sector[sect].add(new_id)

                # Resort DF
                sector_tables[sect]["df"] = sector_tables[sect]["df"].sort_values("abatement_cost").reset_index(drop=True)

                status_msg.value = "✅ Updated."
                refresh_ui()
                # Restore selection of the NEW id
                sel.value = tuple([new_id])

        def on_add_new(_):
            sect = sector_dd.value
            sid = edit_id.value.strip() or "NEW_01"

            # Check duplicate
            df = sector_tables[sect]["df"]
            if sid in set(df["intervention_id"].astype(str)):
                 status_msg.value = "⚠️ Error: ID exists!"
                 return

            new_row = {
                "intervention_id": sid,
                "intervention_name": edit_name.value or "New Measure",
                "abatement_cost": edit_cost.value,
                "abatement_potential": edit_pot.value
            }
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            sector_tables[sect]["df"] = df.sort_values("abatement_cost").reset_index(drop=True)

            selected_ids_by_sector[sect].add(sid)
            custom_ids_by_sector[sect].add(sid)
            status_msg.value = f"✅ Added {sid}"
            refresh_ui()

        def on_delete(_):
            sect = sector_dd.value
            to_del = set(sel.value)
            if not to_del: return

            df = sector_tables[sect]["df"]
            sector_tables[sect]["df"] = df[~df["intervention_id"].astype(str).isin(to_del)].copy().reset_index(drop=True)

            selected_ids_by_sector[sect] -= to_del
            status_msg.value = f"🗑️ Deleted {len(to_del)} items"
            refresh_ui()

        def on_cost_filter_click(_):
            sect = sector_dd.value
            limit = cost_slider.value
            df = sector_tables[sect]["df"]
            # Deselect Items < Limit
            cheap_ids = set(df[df["abatement_cost"] < limit]["intervention_id"].astype(str))
            selected_ids_by_sector[sect] -= cheap_ids
            refresh_ui()

        def on_save(_):
            filtered = {}
            for sect in sector_keys:
                df = sector_tables[sect]["df"].copy()
                keep = selected_ids_by_sector[sect]
                df2 = df[df["intervention_id"].astype(str).isin(keep)].copy().sort_values("abatement_cost").reset_index(drop=True)
                filtered[sect] = {"sheet": sector_tables[sect]["sheet"], "df": df2}

            write_filtered_reference(df_index, filtered, PATHS["ref_filtered"])

            with step2_msg_out:
                clear_output()
                display(widgets.HTML(f"<div class='status-pass'>✅ Saved filtered workbook to: {PATHS['ref_filtered']}</div>"))
                time.sleep(3)
                clear_output()

        # Bindings
        sel.observe(on_select_change, names='value')
        sector_dd.observe(lambda _: refresh_ui(), names='value')
        search.observe(lambda _: refresh_ui(), names='value')

        btn_update.on_click(on_update)
        btn_add_new.on_click(on_add_new)
        btn_delete.on_click(on_delete)
        btn_filter_cost.on_click(on_cost_filter_click)

        btn_all.on_click(lambda _: [selected_ids_by_sector[sector_dd.value].update({v for _,v in sel.options}), refresh_ui()])
        btn_none.on_click(lambda _: [selected_ids_by_sector[sector_dd.value].clear(), refresh_ui()])
        btn_write.on_click(on_save)

        # Layout Assembly
        top_bar = widgets.HBox([sector_dd, search, btn_write], layout=widgets.Layout(justify_content="space-between", margin="0 0 10px 0"))

        list_area = widgets.VBox([
            header, sel,
            widgets.HBox([btn_all, btn_none, cost_slider, btn_filter_cost], layout=widgets.Layout(align_items="center", gap="10px"))
        ])

        edit_area = widgets.VBox([
            widgets.HTML("<b>✏️ Manage Interventions</b>"),
            widgets.HBox([edit_id, edit_name, status_msg]),
            widgets.HBox([edit_cost, edit_pot]),
            widgets.HBox([btn_update, btn_add_new, btn_delete])
        ], layout=widgets.Layout(border="1px dashed #ccc", padding="10px", margin="10px 0"))

        step2_container.children = [top_bar, list_area, plot_out, edit_area, step2_msg_out]
        refresh_ui()

    except Exception as e:
        step2_container.children = [widgets.HTML(f"<b style='color:red'>Error: {str(e)}</b>")]

btn_init = widgets.Button(description="Load & Initialize Filter", button_style="info", icon="filter", layout=widgets.Layout(width="200px"))
btn_init.on_click(init_filter_ui)

step2_box = widgets.VBox([
    widgets.HTML("<h3>🛠️ Step 2: Filter & Edit MACC</h3>"),
    widgets.HTML("Select, edit, or add interventions. The chart updates automatically."),
    btn_init,
    step2_container
], layout=widgets.Layout(padding="10px", border="1px solid #e0e0e0"))

HTML(value='\n<style>\n.macc-list select {\n    font-family: "Courier New", Courier, monospace !important;\n  …

# step 3

In [ ]:
# ======================================================
# STEP 3 — Scale facility MACCs (Corrected)
# ======================================================

# 1. Define UI Elements FIRST so functions can reference them
step3_out = widgets.Output()
step3_progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='info', layout=widgets.Layout(width="98%", visibility="hidden"))
step3_status_label = widgets.Label(value="")

# The missing button definition!
btn_run_scaling = widgets.Button(
    description="Run Scaling",
    button_style="success",
    icon="balance-scale",
    layout=widgets.Layout(width="200px")
)

scale_allow_negative = widgets.Checkbox(value=True, description="Allow negative EI (Go below zero)", indent=False)
scale_add_if_dirty = widgets.Checkbox(value=True, description="Add catch-up intervention if facility is dirtier than reference", indent=False)
scale_strategy = widgets.Dropdown(options=["strict", "full_remaining"], value="strict", description="Strategy:")
scale_addl_offset = widgets.FloatText(value=-10.0, description="Add'l Cost Offset:", style={'description_width': 'initial'})

# ----------------------------
# LOGIC ENGINE
# ----------------------------
def _build_facility_macc_core(facility_prod, facility_ei, ladder_df, config):
    """
    Core math to scale a reference MACC to a specific facility's production and EI.
    """
    ladder = ladder_df.copy().reset_index(drop=True)
    ref_total_delta_ei = float(ladder["delta_ei"].sum())

    # 1. Handle "Dirty" Facilities (EI > Reference Total Potential)
    if config["add_dirty"] and facility_ei > ref_total_delta_ei:
        delta_needed = facility_ei - ref_total_delta_ei
        cheapest_cost = float(ladder["abatement_cost"].min())

        add_row = {
            "intervention_id": "additional_intervention",
            "intervention_name": "Catch-up Intervention (Dirty Facility)",
            "abatement_cost": cheapest_cost + config["offset"],
            "abatement_potential": delta_needed * facility_prod,
            "delta_ei": delta_needed
        }
        ladder = pd.concat([pd.DataFrame([add_row]), ladder], ignore_index=True)
        ladder = ladder.sort_values("abatement_cost").reset_index(drop=True)
        ref_total_delta_ei = float(ladder["delta_ei"].sum())

    # 2. Iterate Ladder
    rows = []
    cum_width = 0.0
    current_ei = float(facility_ei)

    cleaner_than_start = facility_ei < ref_total_delta_ei

    if cleaner_than_start and config["strategy"] == "strict":
        achieved = ref_total_delta_ei - facility_ei
        filtered_ladder = []
        for _, r in ladder.iterrows():
            d = float(r["delta_ei"])
            if achieved >= d - 1e-9:
                achieved -= d
                continue
            if achieved > 0:
                available_d = d - achieved
                achieved = 0
                filtered_ladder.append((r, available_d))
            else:
                filtered_ladder.append((r, d))
    else:
        filtered_ladder = [(r, float(r["delta_ei"])) for _, r in ladder.iterrows()]

    # 3. Build Result Rows
    for r, d_avail in filtered_ladder:
        if d_avail <= 0: continue

        if not config["allow_negative"] and current_ei <= 0:
            break

        d_use = d_avail
        if not config["allow_negative"] and (current_ei - d_use < 0):
            d_use = max(0.0, current_ei)

        if d_use <= 1e-9: continue

        width = facility_prod * d_use
        cum_width += width
        current_ei -= d_use

        rows.append({
            "intervention_id": r["intervention_id"],
            "intervention_name": r["intervention_name"],
            "abatement_cost": float(r["abatement_cost"]),
            "delta_ei": d_use,
            "width_tco2_per_year": width,
            "cumulative_width_tco2_per_year": cum_width,
            "post_intensity_tco2_per_unit": current_ei
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame([{
            "order": 0, "intervention_id": "none", "intervention_name": "no_remaining_interventions",
            "abatement_cost": 0.0, "delta_ei": 0.0, "width_tco2_per_year": 0.0,
            "cumulative_width_tco2_per_year": 0.0, "post_intensity_tco2_per_unit": current_ei
        }])

    out.insert(0, "order", range(1, len(out) + 1))
    return out


def run_scaling(_=None):
    step3_out.clear_output()
    step3_progress.value = 0
    step3_progress.layout.visibility = "visible"
    step3_status_label.value = "Initializing..."
    btn_run_scaling.disabled = True

    try:
        # 1. Validation
        if not Path(PATHS["ref_filtered"]).exists():
            raise FileNotFoundError(f"Missing {PATHS['ref_filtered']}. Please complete Step 2 first.")

        if not Path("facility.xlsx").exists():
             raise FileNotFoundError("Missing facility.xlsx input file.")

        # 2. Load Reference
        step3_status_label.value = "Loading Reference Data..."
        ref_path = Path(PATHS["ref_filtered"])
        df_index = pd.read_excel(ref_path, sheet_name="reference_prod_and_emission")
        df_index = _norm_cols(df_index)

        xls = pd.ExcelFile(ref_path)
        all_sheets = set(xls.sheet_names)

        sector_ref = {}
        for _, r in df_index.iterrows():
            sector = str(r["sector"]).strip().lower()
            prod = float(r["production"])
            sheet = _compose_sector_sheet_name(sector)

            if sheet not in all_sheets: continue

            df = pd.read_excel(ref_path, sheet_name=sheet)
            df = _norm_cols(df)
            df["delta_ei"] = df["abatement_potential"].astype(float) / prod
            df = df.sort_values("abatement_cost").reset_index(drop=True)
            sector_ref[sector] = df

        # 3. Load Facilities
        step3_status_label.value = "Loading Facility Data..."
        df_facilitys = pd.read_excel("facility.xlsx")
        df_facilitys = _norm_cols(df_facilitys)
        req_cols = ["registration_number", "production", "emission_intensity", "sector"]
        _require(df_facilitys, req_cols, "facility.xlsx")

        total_facilities = len(df_facilitys)
        step3_progress.max = total_facilities

        # 4. Processing Loop
        config = {
            "allow_negative": scale_allow_negative.value,
            "strategy": scale_strategy.value,
            "add_dirty": scale_add_if_dirty.value,
            "offset": scale_addl_offset.value
        }

        output_file = PATHS["facility_macc_ccts"]
        writer = pd.ExcelWriter(output_file, engine="xlsxwriter")

        summary_rows = []
        sheet_names_used = set()

        for i, row in df_facilitys.iterrows():
            if i % 5 == 0:
                step3_progress.value = i
                step3_status_label.value = f"Processing {i}/{total_facilities}: {row['registration_number']}"

            reg = str(row["registration_number"]).strip()
            sect = str(row["sector"]).strip().lower()

            if sect not in sector_ref:
                summary_rows.append({"registration_number": reg, "status": "SKIPPED: Sector not found"})
                continue

            macc_table = _build_facility_macc_core(
                float(row["production"]),
                float(row["emission_intensity"]),
                sector_ref[sect],
                config
            )

            sheet_name = safe_excel_sheet_name(reg, sheet_names_used)
            macc_table.to_excel(writer, sheet_name=sheet_name, index=False)

            summary_rows.append({
                "registration_number": reg,
                "sector": sect,
                "sheet": sheet_name,
                "interventions_count": len(macc_table),
                "status": "OK"
            })

        # 5. Finalize
        step3_status_label.value = "Saving Excel file..."
        pd.DataFrame(summary_rows).to_excel(writer, sheet_name="SUMMARY", index=False)
        writer.close()

        step3_progress.value = total_facilities
        step3_progress.bar_style = 'success'
        step3_status_label.value = "Complete!"

        with step3_out:
            display(widgets.HTML(f"<div class='status-pass'>✅ Scaling complete. Output: <b>{output_file}</b> ({len(summary_rows)} facilities processed)</div>"))
            refresh_file_status()

    except Exception as e:
        step3_progress.bar_style = 'danger'
        with step3_out:
            print(f"[ERROR] {str(e)}")
    finally:
        btn_run_scaling.disabled = False

# Bind the event now that the function is defined
btn_run_scaling.on_click(run_scaling)

# ----------------------------
# LAYOUT
# ----------------------------
step3_box = widgets.VBox([
    widgets.HTML("<h3>⚖️ Step 3: Scale Facility MACCs</h3>"),
    widgets.HTML("<p style='color:#666'>Generates unique MACC curves for every facility based on their specific emission intensity and production.</p>"),

    widgets.HBox([
        widgets.VBox([scale_strategy, scale_addl_offset]),
        widgets.VBox([scale_allow_negative, scale_add_if_dirty])
    ], layout=widgets.Layout(justify_content="space-around", border="1px dashed #ccc", padding="10px", margin="10px 0")),

    btn_run_scaling,
    widgets.HBox([step3_progress, step3_status_label], layout=widgets.Layout(margin="10px 0")),
    step3_out
], layout=widgets.Layout(padding="10px", border="1px solid #e0e0e0"))

# step 4

In [ ]:
# ======================================================
# STEP 4 — Build Merged Panel
# ======================================================

# UI Containers
step4_out = widgets.Output()
step4_progress = widgets.IntProgress(value=0, min=0, max=100, description='Progress:', bar_style='info', layout=widgets.Layout(width="98%", visibility="hidden"))
step4_status = widgets.Label(value="")

def clean_join_keys(df, cols):
    """Helper to normalize text columns for merging (uppercase, stripped)."""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.replace(r"\s+", " ", regex=True)
                .str.upper()
            )
    return df

def run_merge(_=None):
    step4_out.clear_output()
    step4_progress.value = 0
    step4_progress.layout.visibility = "visible"
    btn_run_merge.disabled = True

    try:
        # 1. Validation
        step4_status.value = "Validating inputs..."
        if not Path(PATHS["facility_macc_ccts"]).exists():
            raise FileNotFoundError(f"Missing {PATHS['facility_macc_ccts']}. Please complete Step 3 first.")

        # 2. Load Facility Data
        step4_status.value = "Loading Facility CCTS Data..."
        step4_progress.value = 10

        facility_data_ccts = pd.read_excel("facility_data_ccts.xlsx")
        initial_rows = len(facility_data_ccts)

        # Filter for Target EI
        facility_data_ccts = facility_data_ccts[
            facility_data_ccts["TARGET_EI_2025_26_na"].notna() &
            (facility_data_ccts["TARGET_EI_2025_26_na"].astype(str).str.strip() != "")
        ].copy()

        filtered_rows = len(facility_data_ccts)
        dropped_rows = initial_rows - filtered_rows

        # 3. Load Subsector Params
        step4_status.value = "Loading Subsector Parameters..."
        step4_progress.value = 20
        subsector_params = pd.read_excel("subsector_parameters.xlsx")

        # Normalize Subsector Column Names
        sp_cols = {c.lower(): c for c in subsector_params.columns}
        rename_map = {}
        if "sector" in sp_cols: rename_map[sp_cols["sector"]] = "SECTOR"
        if "subsector" in sp_cols: rename_map[sp_cols["subsector"]] = "SUB_SECTOR"
        if "sub_sector" in sp_cols: rename_map[sp_cols["sub_sector"]] = "SUB_SECTOR"
        subsector_params = subsector_params.rename(columns=rename_map)

        # 4. Load & Aggregate MACC Sheets
        step4_status.value = "Reading MACC Sheets (this may take time)..."
        step4_progress.value = 30

        macc_xls = pd.ExcelFile(PATHS["facility_macc_ccts"])
        all_sheets = macc_xls.sheet_names

        # Mapping sheet names to registration numbers via SUMMARY index if possible
        sheet_to_reg = {}
        if "SUMMARY" in all_sheets:
            s = pd.read_excel(macc_xls, sheet_name="SUMMARY")
            s.columns = [c.strip().lower() for c in s.columns]
            if "sheet" in s.columns and "registration_number" in s.columns:
                sheet_to_reg = dict(zip(s["sheet"].astype(str), s["registration_number"].astype(str)))

        valid_regs = set(facility_data_ccts["REGISTRATION_NUMBER"].astype(str).unique())
        frames = []

        # Loop over sheets with progress bar
        sheets_to_process = [sh for sh in all_sheets if sh != "SUMMARY"]
        total_sheets = len(sheets_to_process)

        for idx, sh in enumerate(sheets_to_process):
            # Update progress every 10 sheets to save UI redraw cost
            if idx % 10 == 0:
                perc = 30 + int((idx / total_sheets) * 40) # Map to 30-70% range
                step4_progress.value = perc
                step4_status.value = f"Reading sheet {idx}/{total_sheets}..."

            # Determine Registration Number
            # If map exists use it, else assume sheet name is reg number (fallback)
            reg_id = sheet_to_reg.get(sh, sh)

            # Skip if this facility isn't in our filtered CCTS list
            if str(reg_id) not in valid_regs:
                continue

            df = pd.read_excel(macc_xls, sheet_name=sh)
            df["REGISTRATION_NUMBER"] = reg_id
            frames.append(df)

        step4_status.value = "Concatenating data..."
        facility_macc_long = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

        # 5. Merging
        step4_status.value = "Merging datasets..."
        step4_progress.value = 80

        # Merge 1: Facility Data + MACC
        # Left join to keep facilities even if they have no MACC (though rare)
        facility_with_macc = pd.merge(
            facility_data_ccts,
            facility_macc_long,
            on="REGISTRATION_NUMBER",
            how="left"
        ) if not facility_macc_long.empty else facility_data_ccts.copy()

        # Fix: Normalize Merge Keys for Subsector Join
        facility_with_macc = clean_join_keys(facility_with_macc, ["SECTOR", "SUB_SECTOR"])
        subsector_params = clean_join_keys(subsector_params, ["SECTOR", "SUB_SECTOR"])

        # Merge 2: + Subsector Params
        final_merged = pd.merge(
            facility_with_macc,
            subsector_params,
            on=["SECTOR", "SUB_SECTOR"],
            how="left"
        )

        # 6. Save & Finish
        step4_status.value = "Saving final merged panel..."
        step4_progress.value = 90

        STATE["final_merged"] = final_merged
        final_merged.to_excel(PATHS["merged_panel"], index=False)
        refresh_file_status() # Update Step 1

        step4_progress.value = 100
        step4_progress.bar_style = 'success'
        step4_status.value = "Merge Complete!"

        with step4_out:
            display(widgets.HTML(f"""
            <div class='status-pass'>
                <h3>✅ Merge Successful</h3>
                <ul>
                    <li><b>Source Facilities:</b> {initial_rows}</li>
                    <li><b>Filtered (Valid Targets):</b> {filtered_rows} (Dropped {dropped_rows})</li>
                    <li><b>MACC Data Rows:</b> {len(facility_macc_long)}</li>
                    <li><b>Final Merged Shape:</b> {final_merged.shape}</li>
                    <li><b>Output:</b> {PATHS['merged_panel']}</li>
                </ul>
            </div>
            """))

    except Exception as e:
        step4_progress.bar_style = 'danger'
        step4_status.value = "Error occurred."
        with step4_out:
            print(f"[ERROR] {str(e)}")
            import traceback
            traceback.print_exc()
    finally:
        btn_run_merge.disabled = False

btn_run_merge = widgets.Button(description="Build Merged Panel", button_style="success", icon="table", layout=widgets.Layout(width="200px"))
btn_run_merge.on_click(run_merge)

step4_box = widgets.VBox([
    widgets.HTML("<h3>🧩 Step 4: Build Merged Panel</h3>"),
    widgets.HTML(
        "<p style='color:#666'>Combines <b>Facility Data</b>, <b>Scaled MACC curves</b>, and <b>Subsector Parameters</b> into a single master dataset for simulation.</p>"
    ),
    btn_run_merge,
    widgets.HBox([step4_progress, step4_status], layout=widgets.Layout(margin="10px 0")),
    step4_out
], layout=widgets.Layout(padding="10px", border="1px solid #e0e0e0"))

# step 5

In [ ]:
# ======================================================
# STEP 5 — Scenario table + Run (WIZARD-SAFE, UI/UX UPGRADED)

#

# ✅ Helper text (UX hints) + simple tooltips on buttons
# ✅ Inline validation messages (IDs blank/duplicate, floor>ceiling, bounds invalid, no run selected)
# ✅ Better naming: “Exclude ≤0 cost options” (same semantics as filter_negative_cost)
# ✅ Disable “Run” until valid
# ✅ QoL: Duplicate row (⧉), Reset defaults (confirm)
# ✅ Run summary card BEFORE export
# ✅ Better progress by stage + scrolling log pane
# ✅ Metadata sheet in Excel (timestamp, version, run params, embedded scenario config JSON)
# ✅ Core model functions moved OUTSIDE run_model
# ✅ Centralized constants + explicit PATHS/STATE checks
#
# IMPORTANT:
# - It registers itself for the wizard:
#     globals()["step5_box"] = step5_box
#     globals().setdefault("WIZARD_STEPS", {})["s5"] = step5_box
# ======================================================

import time, json, re
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

# -----------------------------
# Versioning / constants
# -----------------------------
#STEP5_VERSION = "step5_uiux_v5.2_wizard_safe_2026-02-17"

STEP5_VERSION = "v1"

REQUIRED_COLS = [
    "REGISTRATION_NUMBER","SECTOR","SUB_SECTOR",
    "PROD_BASELINE","BASELINE_EI","TARGET_EI_2025_26_na",
    "abatement_cost","delta_ei","u_baseline","epsilon"
]

DEFAULT_SCENARIOS = [
    {"id":"A",      "name":"Scenario A: 1% Autonomous Technical Progress", "description":"baseline_ei × 0.99", "delta_ei_factor":1.0, "baseline_ei_factor":0.99, "run": True},
    {"id":"A_pf5",  "name":"Scenario A + Price Floor $5",  "description":"A with floor 5",  "delta_ei_factor":1.0, "baseline_ei_factor":0.99, "price_floor":5,  "run": True},
    {"id":"A_pf10", "name":"Scenario A + Price Floor $10", "description":"A with floor 10", "delta_ei_factor":1.0, "baseline_ei_factor":0.99, "price_floor":10, "run": True},
    {"id":"B",      "name":"Scenario B: Baseline", "description":"baseline", "delta_ei_factor":1.0, "baseline_ei_factor":1.0, "run": True},
    {"id":"C",      "name":"Scenario C: Slow Technology Adoption", "description":"delta_ei × 0.3", "delta_ei_factor":0.3, "baseline_ei_factor":1.0, "run": True},
    {"id":"C_pc15", "name":"Scenario C + Price Ceiling $15", "description":"C with ceiling 15", "delta_ei_factor":0.3, "baseline_ei_factor":1.0, "price_ceiling":15, "run": True},
    {"id":"C_pc20", "name":"Scenario C + Price Ceiling $20", "description":"C with ceiling 20", "delta_ei_factor":0.3, "baseline_ei_factor":1.0, "price_ceiling":20, "run": True},
]

SHEET_NAMES = {
    "summary":   "Summary",
    "metadata":  "Metadata",
    "facility":  "facility_Level_All_Data",
    "sub_total": "Subsector_Avg_Total_Cost",
    "sub_tech":  "Subsector_Avg_Tech_Cost",
    "sub_carb":  "Subsector_Avg_Carbon_Cost",
    "sub_ratio": "Subsector_Cost_Ratio_Pct",
    "sub_emis":  "Subsector_Emission_Red_Pct",
    "sub_int":   "Subsector_Intensity_Change",
    "sub_prod":  "Subsector_Production_Change",
}

_invalid_sheet_chars = re.compile(r"[\[\]\:\*\?\/\\]")

def safe_sheet_name(name: str, used: set):
    base = _invalid_sheet_chars.sub("_", str(name)).strip()
    base = base[:31] if len(base) > 31 else base
    if base == "":
        base = "Sheet"
    cand = base
    k = 1
    while cand in used:
        suffix = f"_{k}"
        cand = (base[:31-len(suffix)] + suffix) if len(base)+len(suffix) > 31 else (base + suffix)
        k += 1
    used.add(cand)
    return cand

def validate_final_merged(df: pd.DataFrame):
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"final_merged missing required columns: {missing}")

# -----------------------------
# Core model functions (outside run_model)
# -----------------------------
def prepare_math_data_fast(df, delta_ei_factor=1.0, baseline_ei_factor=1.0, filter_negative_cost=True):
    """
    Returns:
      para: (n,5) [q0,e_b,e_t,py,ep]
      tech_list: list per facility with {costs,cum_delta,cum_costdelta}
      info: (n,3) [facility_id, sector, subsector]
      prep_stats: dict for logging
    """
    t0 = df.dropna(subset=["TARGET_EI_2025_26_na"]).copy()

    t0["abatement_cost"] = pd.to_numeric(t0["abatement_cost"], errors="coerce")
    t0["delta_ei"]       = pd.to_numeric(t0["delta_ei"], errors="coerce") * float(delta_ei_factor)
    t0["BASELINE_EI"]    = pd.to_numeric(t0["BASELINE_EI"], errors="coerce") * float(baseline_ei_factor)
    t0["PROD_BASELINE"]  = pd.to_numeric(t0["PROD_BASELINE"], errors="coerce").fillna(0.0)
    t0["TARGET_EI_2025_26_na"] = pd.to_numeric(t0["TARGET_EI_2025_26_na"], errors="coerce").fillna(0.0)
    t0["u_baseline"]     = pd.to_numeric(t0["u_baseline"], errors="coerce").fillna(0.0)
    t0["epsilon"]        = pd.to_numeric(t0["epsilon"], errors="coerce").fillna(0.0)

    t0 = t0.sort_values(["REGISTRATION_NUMBER", "abatement_cost"], kind="mergesort")

    ids = t0["REGISTRATION_NUMBER"].astype(str).values
    sectors = t0["SECTOR"].values
    subsectors = t0["SUB_SECTOR"].values

    costs  = t0["abatement_cost"].to_numpy(dtype=float)
    deltas = t0["delta_ei"].to_numpy(dtype=float)

    q0_arr = t0["PROD_BASELINE"].to_numpy(dtype=float)
    eb_arr = t0["BASELINE_EI"].to_numpy(dtype=float)
    et_arr = t0["TARGET_EI_2025_26_na"].to_numpy(dtype=float)
    py_arr = t0["u_baseline"].to_numpy(dtype=float)
    ep_arr = t0["epsilon"].to_numpy(dtype=float)

    if ids.size == 0:
        raise ValueError("No rows remaining after filtering TARGET_EI_2025_26_na.")

    _, idx_start, counts = np.unique(ids, return_index=True, return_counts=True)

    para_list, info_list, tech_list = [], [], []
    dropped_nan = dropped_nonpos_delta = dropped_nonpos_cost = kept_options = 0

    for i0, cnt in zip(idx_start, counts):
        sl = slice(i0, i0+cnt)

        info_list.append([ids[i0], sectors[i0], subsectors[i0]])
        para_list.append([q0_arr[i0], eb_arr[i0], et_arr[i0], py_arr[i0], ep_arr[i0]])

        c = costs[sl]
        d = deltas[sl]

        mask = np.isfinite(c) & np.isfinite(d)
        dropped_nan += int((~mask).sum())

        mask2 = mask & (d > 0)
        dropped_nonpos_delta += int((mask & ~(d > 0)).sum())

        if filter_negative_cost:
            mask3 = mask2 & (c > 0)  # semantics unchanged: drop cost<=0
            dropped_nonpos_cost += int((mask2 & ~(c > 0)).sum())
        else:
            mask3 = mask2

        c = c[mask3]
        d = d[mask3]

        if c.size == 0:
            tech_list.append({"costs": np.empty(0), "cum_delta": np.empty(0), "cum_costdelta": np.empty(0)})
        else:
            tech_list.append({
                "costs": c,
                "cum_delta": np.cumsum(d),
                "cum_costdelta": np.cumsum(c * d),
            })
            kept_options += int(c.size)

    prep_stats = {
        "rows_input_after_target_filter": int(t0.shape[0]),
        "facilities": int(len(tech_list)),
        "options_kept_total": int(kept_options),
        "dropped_nan": int(dropped_nan),
        "dropped_nonpos_delta": int(dropped_nonpos_delta),
        "dropped_nonpos_cost": int(dropped_nonpos_cost),
        "filter_negative_cost": bool(filter_negative_cost),
    }

    return np.array(para_list, dtype=float), tech_list, np.array(info_list, dtype=object), prep_stats

def _compute_aj_tc(tech_list, pc: float):
    n = len(tech_list)
    aj = np.zeros(n, dtype=float)
    tc = np.zeros(n, dtype=float)
    for i, T in enumerate(tech_list):
        costs = T["costs"]
        if costs.size == 0:
            continue
        idx = np.searchsorted(costs, pc, side="right") - 1
        if idx >= 0:
            aj[i] = T["cum_delta"][idx]
            tc[i] = T["cum_costdelta"][idx]
    return aj, tc

def eval_market(para, tech_list, pc: float):
    """Fast evaluation (no DataFrame): returns (imbalance, total_buy, total_sell)."""
    q0, e_b, e_t, py, ep = para[:,0], para[:,1], para[:,2], para[:,3], para[:,4]
    rint = e_b - e_t

    aj, tc = _compute_aj_tc(tech_list, float(pc))

    unit_carbon_cost_change = float(pc) * (rint - aj)
    del_u = unit_carbon_cost_change + tc

    delta = np.zeros_like(py, dtype=float)
    mask_py = (py > 0) & np.isfinite(py)
    delta[mask_py] = del_u[mask_py] / py[mask_py]

    d_del = np.maximum(1 - delta, 1e-6)
    qst = q0 * (d_del ** ep)

    emis  = qst * np.maximum(e_b - aj, 0)
    allow = qst * e_t
    ncap  = emis - allow

    total_buy = float(np.sum(ncap[ncap > 0]))
    total_sel = float(-np.sum(ncap[ncap < 0]))
    imbalance = total_buy - total_sel
    return imbalance, total_buy, total_sel

def build_facility_table(para, tech_list, info, pc: float):
    """Full facility table at a given price (used once per scenario at equilibrium)."""
    q0, e_b, e_t, py, ep = para[:,0], para[:,1], para[:,2], para[:,3], para[:,4]
    rint = e_b - e_t

    aj, tc = _compute_aj_tc(tech_list, float(pc))

    unit_carbon_cost_change = float(pc) * (rint - aj)
    del_u = unit_carbon_cost_change + tc

    delta = np.zeros_like(py, dtype=float)
    mask_py = (py > 0) & np.isfinite(py)
    delta[mask_py] = del_u[mask_py] / py[mask_py]

    d_del = np.maximum(1 - delta, 1e-6)
    qst = q0 * (d_del ** ep)

    emis = qst * np.maximum(e_b - aj, 0)
    allow = qst * e_t
    ncap = emis - allow
    emis_baseline = q0 * e_b

    total_tech_cost = tc * qst
    carbon_market_cost = float(pc) * ncap

    tbl = pd.DataFrame({
        "facility_id": info[:,0],
        "sector": info[:,1],
        "subsector": info[:,2],
        "q0": q0, "qst": qst, "py": py, "epsilon": ep,
        "emi_base": e_b, "emi_targ": e_t, "r_int": rint,
        "AJ": aj,
        "unit_tech_cost": tc,
        "unit_carbon_cost_change": unit_carbon_cost_change,
        "del_u": del_u,
        "total_tech_cost": total_tech_cost,
        "carbon_market_cost": carbon_market_cost,
        "Emis": emis,
        "Emis_baseline": emis_baseline,
        "Allowance": allow,
        "N": ncap,
        "Pc": float(pc),
        "output_change_pct": np.where(q0 > 0, ((qst - q0) / q0) * 100, 0.0),
    })
    imb, td, ts = eval_market(para, tech_list, float(pc))
    return imb, tbl, td, ts

def subsector_analysis(facility_tbl: pd.DataFrame):
    def wavg(x, w):
        w = np.asarray(w, dtype=float)
        x = np.asarray(x, dtype=float)
        ws = np.sum(w[np.isfinite(w)])
        if ws > 0:
            m = np.isfinite(x) & np.isfinite(w)
            if np.any(m):
                return float(np.average(x[m], weights=w[m]))
        return float(np.nanmean(x)) if x.size else 0.0

    rows = []
    grp = facility_tbl.groupby(["sector","subsector"], dropna=False)
    for (sec, subsec), g in grp:
        qst_w = g["qst"].to_numpy(dtype=float)
        rows.append({
            "sector": sec,
            "subsector": subsec,
            "num_facilitys": int(g["facility_id"].count()),
            "production_baseline": float(g["q0"].sum()),
            "production_with_ETS": float(g["qst"].sum()),
            "avg_product_price": wavg(g["py"], qst_w),
            "emission_baseline": float(g["Emis_baseline"].sum()),
            "emission_with_ETS": float(g["Emis"].sum()),
            "net_demand": float(g["N"].sum()),
            "avg_unit_total_cost_change": wavg(g["del_u"], qst_w),
            "avg_unit_tech_cost": wavg(g["unit_tech_cost"], qst_w),
            "avg_unit_carbon_cost": wavg(g["unit_carbon_cost_change"], qst_w),
            "total_tech_cost_ETS": float(g["total_tech_cost"].sum()),
            "total_net_carbon_cost_ETS": float(g["carbon_market_cost"].sum()),
        })
    sub = pd.DataFrame(rows)

    sub["unit_cost_change_ratio"] = np.where(
        sub["avg_product_price"] > 0,
        (sub["avg_unit_total_cost_change"] / sub["avg_product_price"]) * 100,
        0.0
    )
    sub["emission_reduction_pct"] = np.where(
        sub["emission_baseline"] > 0,
        ((sub["emission_baseline"] - sub["emission_with_ETS"]) / sub["emission_baseline"]) * 100,
        0.0
    )
    sub["production_change_pct"] = np.where(
        sub["production_baseline"] > 0,
        ((sub["production_with_ETS"] - sub["production_baseline"]) / sub["production_baseline"]) * 100,
        0.0
    )
    sub["carbon_intensity_baseline"] = np.where(
        sub["production_baseline"] > 0,
        sub["emission_baseline"] / sub["production_baseline"],
        0.0
    )
    sub["carbon_intensity_with_ETS"] = np.where(
        sub["production_with_ETS"] > 0,
        sub["emission_with_ETS"] / sub["production_with_ETS"],
        0.0
    )
    sub["carbon_intensity_change_pct"] = np.where(
        sub["carbon_intensity_baseline"] > 0,
        ((sub["carbon_intensity_with_ETS"] - sub["carbon_intensity_baseline"]) / sub["carbon_intensity_baseline"]) * 100,
        0.0
    )
    return sub

def analyze_sd_curve_fast(para, tech_list, price_min: float, price_cap: float, sd_step: float,
                          cancel_cb=None, progress_cb=None):
    p0 = float(price_min)
    p1 = float(price_cap)
    st = max(float(sd_step), 1e-9)
    pr = np.arange(p0, p1 + st, st)
    rows = []
    for k, pc in enumerate(pr):
        if cancel_cb and (k % 25 == 0):
            cancel_cb(f"(SD curve @ {pc:.2f})")
        if progress_cb and (k % 50 == 0):
            progress_cb(k, len(pr), float(pc))
        imb, td, ts = eval_market(para, tech_list, float(pc))
        rows.append({"carbon_price": float(pc), "total_demand": td, "total_supply": ts, "imbalance": float(imb)})
    return pd.DataFrame(rows)

def find_equilibrium_price(para, tech_list, floor=None, ceiling=None,
                           price_min=0.0, price_cap=50.0, price_step=0.01,
                           cancel_cb=None, status_cb=None):
    min_cap = float(price_min)
    max_cap = float(price_cap)
    step = max(float(price_step), 1e-6)

    lo_bound = min_cap if floor is None else max(min_cap, float(floor))
    hi_bound = max_cap if ceiling is None else min(max_cap, float(ceiling))
    if lo_bound > hi_bound:
        raise ValueError(f"Search bounds invalid after floor/ceiling: lo={lo_bound}, hi={hi_bound}")

    # upper bound speed helper: max observed tech cost (never below lo_bound)
    eff_max_cost = lo_bound
    for T in tech_list:
        c = T["costs"]
        if c.size:
            m = float(np.nanmax(c))
            if np.isfinite(m):
                eff_max_cost = max(eff_max_cost, m)
    hi = min(hi_bound, eff_max_cost)
    if lo_bound > hi:
        hi = lo_bound

    coarse_step = max(step * 50.0, 0.5)

    def _eval(pc):
        imb, _, _ = eval_market(para, tech_list, float(pc))
        return float(imb)

    prices = np.arange(lo_bound, hi + coarse_step, coarse_step)
    best_pc = float(prices[0])
    best_abs = float("inf")

    bracket = None
    prev_p, prev_f = None, None

    for k, pc in enumerate(prices):
        if cancel_cb and (k % 50 == 0):
            cancel_cb("(equilibrium coarse)")
        f = float(_eval(float(pc)))
        a = abs(f)
        if a < best_abs:
            best_abs = a
            best_pc = float(pc)

        if status_cb and (k % 100 == 0):
            status_cb(f"eq search (coarse) {k:,}/{len(prices):,} — pc={pc:.2f} (best={best_pc:.2f})")

        if prev_f is not None and (f == 0.0 or (prev_f > 0 and f < 0) or (prev_f < 0 and f > 0)):
            bracket = (float(prev_p), float(prev_f), float(pc), float(f))
            break

        prev_p, prev_f = float(pc), float(f)

    if bracket is not None:
        pL, fL, pR, fR = bracket
        for _ in range(80):
            if cancel_cb:
                cancel_cb("(equilibrium bisection)")
            mid = 0.5 * (pL + pR)
            fM = float(_eval(mid))
            if abs(fM) < best_abs:
                best_abs = abs(fM)
                best_pc = mid
            if (pR - pL) <= step:
                break
            if (fL > 0 and fM < 0) or (fL < 0 and fM > 0):
                pR, fR = mid, fM
            else:
                pL, fL = mid, fM
        return float(best_pc)

    lo = max(lo_bound, best_pc - coarse_step)
    hi2 = min(hi, best_pc + coarse_step)
    fine = np.arange(lo, hi2 + step, step)
    for k, pc in enumerate(fine):
        if cancel_cb and (k % 200 == 0):
            cancel_cb("(equilibrium refine)")
        f = float(_eval(float(pc)))
        a = abs(f)
        if a < best_abs:
            best_abs = a
            best_pc = float(pc)
    return float(best_pc)

# -----------------------------
# UI helpers
# -----------------------------
def fmt_elapsed(sec):
    sec = int(sec)
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

def fmt_eta(sec):
    if sec is None:
        return "—"
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f"{h}h {m}m {s}s"
    if m > 0:
        return f"{m}m {s}s"
    return f"{s}s"

# -----------------------------
# Step 5 widgets
# -----------------------------
step5_out = widgets.Output()

# Log pane (scrolling)
log_pane = widgets.Textarea(
    value="",
    description="Log:",
    layout=widgets.Layout(width="100%", height="180px"),
    disabled=True
)
def log(msg, level="INFO"):
    ts = datetime.now().strftime("%H:%M:%S")
    log_pane.value = (log_pane.value + f"[{ts}] [{level}] {msg}\n")[-20000:]

# Inline validation messages
validation_html = widgets.HTML(value="")

def set_validation(ok: bool, messages):
    if ok:
        validation_html.value = "<div style='color:#2e7d32; font-weight:700;'>✓ Ready to run</div>"
    else:
        items = "".join([f"<li>{m}</li>" for m in messages]) if messages else "<li>Invalid configuration</li>"
        validation_html.value = (
            "<div style='color:#b71c1c; font-weight:800;'>Fix these before running:</div>"
            f"<ul style='color:#b71c1c; margin:6px 0 0 18px; padding:0;'>{items}</ul>"
        )

# Cancel flag
CANCEL = {"stop": False}

# Progress widgets
progress_bar = widgets.IntProgress(value=0, min=0, max=1, description="Scenarios:", bar_style="info", layout=widgets.Layout(width="98%"))
stage_bar = widgets.IntProgress(value=0, min=0, max=6, description="Stage:", bar_style="info", layout=widgets.Layout(width="98%"))

timer_html = widgets.HTML(value="<b>Elapsed:</b> 00:00:00")
eta_html = widgets.HTML(value="<b>ETA remaining:</b> —")
progress_html = widgets.HTML(value="<b>Status:</b> idle")

# Summary card (pre-export)
summary_card = widgets.HTML(value="")

# Run controls
price_step = widgets.FloatText(value=0.01, description="Eq step:", layout=widgets.Layout(width="200px"))
price_min  = widgets.FloatText(value=0.0,  description="Min cap:", layout=widgets.Layout(width="200px"))
price_cap  = widgets.FloatText(value=50.0, description="Max cap:", layout=widgets.Layout(width="200px"))
sd_step    = widgets.FloatText(value=1.0,  description="SD step:", layout=widgets.Layout(width="200px"))

helper_run = widgets.HTML(
    "<div style='color:#555; font-size:12px; line-height:1.35;'>"
    "<b>Notes:</b> Eq step controls equilibrium search precision. SD step controls SD-curve point density. "
    "Very small steps can make runs slow.</div>"
)

# -----------------------------
# Scenario editor (Row UI)
# -----------------------------
scenario_rows = []
scenario_rows_box = widgets.VBox()

def _make_unique_id(existing_ids, base):
    base = (base or "S").strip()
    if base == "":
        base = "S"
    cand = base
    k = 1
    while cand in existing_ids:
        cand = f"{base}_{k}"
        k += 1
    return cand

def make_scenario_row(init=None):
    init = init or {}

    sel_del = widgets.Checkbox(value=False, layout=widgets.Layout(width="28px"))

    btn_dup = widgets.Button(description="⧉", tooltip="Duplicate this scenario", layout=widgets.Layout(width="34px"))

    run_flag = widgets.Checkbox(
        value=bool(init.get("run", True)),
        description="run",
        indent=False,
        layout=widgets.Layout(width="60px")
    )

    sid  = widgets.Text(value=str(init.get("id","")), layout=widgets.Layout(width="110px"))
    name = widgets.Text(value=str(init.get("name","")), layout=widgets.Layout(width="260px"))
    desc = widgets.Text(value=str(init.get("description","")), layout=widgets.Layout(width="340px"))

    delta = widgets.FloatText(value=float(init.get("delta_ei_factor",1.0)), layout=widgets.Layout(width="120px"))
    base  = widgets.FloatText(value=float(init.get("baseline_ei_factor",1.0)), layout=widgets.Layout(width="120px"))

    use_floor = widgets.Checkbox(value=init.get("price_floor") is not None, description="floor", indent=False, layout=widgets.Layout(width="70px"))
    floor_val = widgets.FloatText(value=float(init.get("price_floor",0.0) or 0.0), layout=widgets.Layout(width="100px"))

    use_ceil = widgets.Checkbox(value=init.get("price_ceiling") is not None, description="ceiling", indent=False, layout=widgets.Layout(width="85px"))
    ceil_val = widgets.FloatText(value=float(init.get("price_ceiling",0.0) or 0.0), layout=widgets.Layout(width="100px"))

    # Better naming (same semantics as old filter_negative_cost)
    neg = widgets.Checkbox(
        value=bool(init.get("filter_negative_cost",True)),
        description="Exclude ≤0 cost options",
        indent=False,
        layout=widgets.Layout(width="220px")
    )

    floor_val.disabled = not use_floor.value
    ceil_val.disabled = not use_ceil.value
    use_floor.observe(lambda ch: setattr(floor_val, "disabled", not ch["new"]), names="value")
    use_ceil.observe(lambda ch: setattr(ceil_val, "disabled", not ch["new"]), names="value")

    row = widgets.HBox(
        [sel_del, btn_dup, run_flag, sid, name, desc, delta, base, use_floor, floor_val, use_ceil, ceil_val, neg],
        layout=widgets.Layout(flex_flow="row wrap", gap="8px", width="100%")
    )

    pack = {
        "ui": row,
        "sel_del": sel_del,
        "dup_btn": btn_dup,
        "run": run_flag,
        "id": sid,
        "name": name,
        "desc": desc,
        "delta": delta,
        "base": base,
        "use_floor": use_floor,
        "floor": floor_val,
        "use_ceil": use_ceil,
        "ceil": ceil_val,
        "neg": neg
    }

    # Duplicate
    def _do_dup(_btn=None, _pack=pack):
        existing = set([(r["id"].value or "").strip() for r in scenario_rows])
        src_id = (_pack["id"].value or "S").strip()
        new_id = _make_unique_id(existing, src_id + "_copy")
        scenario_rows.append(make_scenario_row({
            "id": new_id,
            "run": bool(_pack["run"].value),
            "name": str(_pack["name"].value),
            "description": str(_pack["desc"].value),
            "delta_ei_factor": float(_pack["delta"].value),
            "baseline_ei_factor": float(_pack["base"].value),
            "price_floor": (float(_pack["floor"].value) if _pack["use_floor"].value else None),
            "price_ceiling": (float(_pack["ceil"].value) if _pack["use_ceil"].value else None),
            "filter_negative_cost": bool(_pack["neg"].value),
        }))
        redraw_scenarios()
        update_validation()

    btn_dup.on_click(_do_dup)

    # Any edit triggers validation
    def _on_any_change(_):
        update_validation()

    for w in [run_flag, sid, name, desc, delta, base, use_floor, floor_val, use_ceil, ceil_val, neg]:
        w.observe(_on_any_change, names="value")

    return pack

def redraw_scenarios():
    scenario_rows_box.children = [r["ui"] for r in scenario_rows]

def load_default_scenarios():
    scenario_rows.clear()
    for sc in DEFAULT_SCENARIOS:
        scenario_rows.append(make_scenario_row(sc))
    redraw_scenarios()

# Buttons
btn_reset_defaults = widgets.Button(description="Reset to defaults", button_style="warning", tooltip="Reset scenarios to default set")
btn_add = widgets.Button(description="Add row", button_style="success")
btn_delete = widgets.Button(description="Delete selected", button_style="danger")
btn_run_all = widgets.Button(description="Run: select all")
btn_run_none = widgets.Button(description="Run: select none")

cfg_name = widgets.Text(value="scenarios_config.json", description="Config:", layout=widgets.Layout(width="340px"))
btn_save_cfg = widgets.Button(description="Save config")
btn_load_cfg = widgets.Button(description="Load config")
cfg_out = widgets.Output()

RESET_ARM = {"armed": False, "ts": 0.0}

def reset_defaults(_=None):
    now = time.time()
    if (not RESET_ARM["armed"]) or (now - RESET_ARM["ts"] > 6):
        RESET_ARM["armed"] = True
        RESET_ARM["ts"] = now
        btn_reset_defaults.description = "Confirm reset (click again)"
        btn_reset_defaults.button_style = "danger"
        log("Reset armed: click again within 6 seconds to confirm.", "WARN")
        update_validation()
        return
    RESET_ARM["armed"] = False
    btn_reset_defaults.description = "Reset to defaults"
    btn_reset_defaults.button_style = "warning"
    load_default_scenarios()
    log("Defaults loaded.", "INFO")
    update_validation()

def add_scenario(_=None):
    existing = set([(r["id"].value or "").strip() for r in scenario_rows])
    new_id = _make_unique_id(existing, f"S{len(scenario_rows)+1}")
    scenario_rows.append(make_scenario_row({
        "id": new_id,
        "name": "New scenario",
        "description": "",
        "delta_ei_factor": 1.0,
        "baseline_ei_factor": 1.0,
        "run": True
    }))
    redraw_scenarios()
    update_validation()

def delete_selected(_=None):
    kept = [r for r in scenario_rows if not r["sel_del"].value]
    scenario_rows.clear()
    scenario_rows.extend(kept)
    redraw_scenarios()
    update_validation()

def run_select_all(_=None):
    for r in scenario_rows:
        r["run"].value = True
    update_validation()

def run_select_none(_=None):
    for r in scenario_rows:
        r["run"].value = False
    update_validation()

btn_reset_defaults.on_click(reset_defaults)
btn_add.on_click(add_scenario)
btn_delete.on_click(delete_selected)
btn_run_all.on_click(run_select_all)
btn_run_none.on_click(run_select_none)

def get_all_scenarios_from_ui():
    scenarios = {}
    order = []
    seen = set()

    for r in scenario_rows:
        sid = (r["id"].value or "").strip()
        if sid == "":
            raise ValueError("Scenario ID cannot be blank.")
        if sid in seen:
            raise ValueError(f"Duplicate scenario ID: {sid}")
        seen.add(sid)

        floor = float(r["floor"].value) if r["use_floor"].value else None
        ceil  = float(r["ceil"].value)  if r["use_ceil"].value else None
        if (floor is not None) and (ceil is not None) and (floor > ceil):
            raise ValueError(f"Scenario {sid}: floor ({floor}) cannot exceed ceiling ({ceil}).")

        scenarios[sid] = {
            "run": bool(r["run"].value),
            "name": (r["name"].value or "").strip(),
            "description": (r["desc"].value or "").strip(),
            "delta_ei_factor": float(r["delta"].value),
            "baseline_ei_factor": float(r["base"].value),
            "price_floor": floor,
            "price_ceiling": ceil,
            "filter_negative_cost": bool(r["neg"].value),
        }
        order.append(sid)

    if not order:
        raise ValueError("No scenarios.")
    return scenarios, order

def get_scenarios_to_run_from_ui():
    all_sc, all_order = get_all_scenarios_from_ui()
    run_order = [sid for sid in all_order if all_sc[sid]["run"]]
    run_sc = {sid: all_sc[sid] for sid in run_order}
    if not run_order:
        raise ValueError("No scenarios selected to run. Check the 'run' boxes.")
    return run_sc, run_order, all_sc, all_order

def save_cfg(_=None):
    with cfg_out:
        clear_output()
        try:
            sc, order = get_all_scenarios_from_ui()
            Path(cfg_name.value).write_text(json.dumps({"order": order, "scenarios": sc}, indent=2))
            print(f"[OK] Saved {cfg_name.value}")
        except Exception as e:
            print(f"[ERROR] {e}")

def load_cfg(_=None):
    with cfg_out:
        clear_output()
        try:
            p = Path(cfg_name.value)
            if not p.exists():
                print("[ERROR] Config not found.")
                return
            payload = json.loads(p.read_text())
            order = payload["order"]
            scs = payload["scenarios"]

            scenario_rows.clear()
            for sid in order:
                sc = scs[sid]
                scenario_rows.append(make_scenario_row({
                    "id": sid,
                    "run": sc.get("run", True),
                    "name": sc.get("name",""),
                    "description": sc.get("description",""),
                    "delta_ei_factor": sc.get("delta_ei_factor",1.0),
                    "baseline_ei_factor": sc.get("baseline_ei_factor",1.0),
                    "price_floor": sc.get("price_floor", None),
                    "price_ceiling": sc.get("price_ceiling", None),
                    "filter_negative_cost": sc.get("filter_negative_cost", True),
                }))
            redraw_scenarios()
            print(f"[OK] Loaded {cfg_name.value}")
            update_validation()
        except Exception as e:
            print(f"[ERROR] {e}")

btn_save_cfg.on_click(save_cfg)
btn_load_cfg.on_click(load_cfg)

# -----------------------------
# Enable/disable UI during run
# -----------------------------
def set_scenario_ui_enabled(enabled: bool):
    for b in [btn_reset_defaults, btn_add, btn_delete, btn_run_all, btn_run_none, btn_save_cfg, btn_load_cfg]:
        b.disabled = not enabled
    cfg_name.disabled = not enabled
    price_step.disabled = not enabled
    price_min.disabled = not enabled
    price_cap.disabled = not enabled
    sd_step.disabled = not enabled

    for r in scenario_rows:
        for k in ["sel_del","dup_btn","run","id","name","desc","delta","base","use_floor","floor","use_ceil","ceil","neg"]:
            r[k].disabled = not enabled
        r["floor"].disabled = (not enabled) or (not r["use_floor"].value)
        r["ceil"].disabled  = (not enabled) or (not r["use_ceil"].value)

# -----------------------------
# Inline validation + gate Run button
# -----------------------------
btn_run_model = widgets.Button(description="Run scenarios + export results workbook", button_style="success")
btn_cancel = widgets.Button(description="Cancel run", button_style="danger")
btn_cancel.disabled = True

def cancel_run(_=None):
    CANCEL["stop"] = True
    progress_html.value = "<b>Status:</b> Cancel requested… stopping after current operation."
btn_cancel.on_click(cancel_run)

def _validate_ui_config():
    msgs = []
    # scenarios validation
    try:
        sc_all, order_all = get_all_scenarios_from_ui()
    except Exception as e:
        msgs.append(str(e))
        return False, msgs

    # bounds validation
    if not np.isfinite(price_step.value) or price_step.value <= 0:
        msgs.append("Eq step must be > 0.")
    if not np.isfinite(sd_step.value) or sd_step.value <= 0:
        msgs.append("SD step must be > 0.")
    if not np.isfinite(price_min.value):
        msgs.append("Min cap must be numeric.")
    if not np.isfinite(price_cap.value):
        msgs.append("Max cap must be numeric.")
    if np.isfinite(price_min.value) and np.isfinite(price_cap.value) and (price_min.value > price_cap.value):
        msgs.append("Min cap cannot exceed Max cap.")

    # must have at least one run=true
    if not any(sc_all[sid]["run"] for sid in order_all):
        msgs.append("Select at least one scenario to run (check 'run').")

    ok = len(msgs) == 0 and (len(order_all) > 0)
    return ok, msgs

def update_validation():
    # auto-disarm reset after 6 sec
    if RESET_ARM["armed"] and (time.time() - RESET_ARM["ts"] > 6):
        RESET_ARM["armed"] = False
        btn_reset_defaults.description = "Reset to defaults"
        btn_reset_defaults.button_style = "warning"

    ok, msgs = _validate_ui_config()
    set_validation(ok, msgs)
    if not btn_run_model.description.startswith("Running"):
        btn_run_model.disabled = not ok

for w in [price_step, price_min, price_cap, sd_step]:
    w.observe(lambda ch: update_validation(), names="value")

# -----------------------------
# Run model (stage progress + log + summary card + metadata)
# -----------------------------
def run_model(_=None):
    with step5_out:
        clear_output()

        start_time = time.time()
        scenario_times = []

        def compute_eta(remaining):
            if len(scenario_times) < 2:
                return None
            avg = sum(scenario_times) / len(scenario_times)
            return avg * remaining + avg

        def update_status(msg, stage="", current_idx=None, total=None, stage_i=None):
            timer_html.value = f"<b>Elapsed:</b> {fmt_elapsed(time.time() - start_time)}"
            if current_idx is not None and total is not None:
                remaining = max(0, total - (current_idx + 1))
                eta_html.value = f"<b>ETA remaining:</b> {fmt_eta(compute_eta(remaining))}"
            else:
                eta_html.value = "<b>ETA remaining:</b> —"
            progress_html.value = f"<b>Status:</b> {msg}" + (f" <span style='color:#555'>(stage: {stage})</span>" if stage else "")
            if stage_i is not None:
                stage_bar.value = int(stage_i)

        def check_cancel(where=""):
            if CANCEL["stop"]:
                update_status(f"Cancelled by user. {where}", stage="cancel", stage_i=0)
                log(f"Cancelled by user {where}", "WARN")
                raise KeyboardInterrupt("Cancelled by user")

        try:
            ok, msgs = _validate_ui_config()
            if not ok:
                set_validation(False, msgs)
                log("Cannot run: configuration invalid.", "ERROR")
                return

            # Explicit PATHS/STATE checks
            if "PATHS" not in globals() or not isinstance(globals().get("PATHS"), dict):
                log("PATHS dict not found. Run earlier steps (where PATHS is defined).", "ERROR")
                print("[ERROR] PATHS not found. Please run earlier steps where PATHS is defined.")
                return
            if "STATE" not in globals() or not isinstance(globals().get("STATE"), dict):
                globals()["STATE"] = {}
                log("STATE dict not found; created empty STATE={}.", "WARN")

            for k in ["merged_panel", "scenario_results"]:
                if k not in PATHS:
                    log(f"PATHS missing key: {k}", "ERROR")
                    print(f"[ERROR] PATHS missing key: {k}")
                    return

            # Reset cancel, enable cancel
            CANCEL["stop"] = False
            btn_cancel.disabled = False

            # Disable UI
            btn_run_model.disabled = True
            btn_run_model.description = "Running..."
            btn_run_model.button_style = "warning"
            set_scenario_ui_enabled(False)

            # Reset visuals
            progress_bar.bar_style = "info"
            stage_bar.bar_style = "info"
            summary_card.value = ""
            progress_bar.value = 0
            stage_bar.value = 0

            # Load merged data
            update_status("Loading merged data...", stage="load", stage_i=0)
            log("Loading final_merged...", "INFO")

            if STATE.get("final_merged") is None:
                mp = Path(PATHS["merged_panel"])
                if mp.exists():
                    STATE["final_merged"] = pd.read_excel(PATHS["merged_panel"])
                    log(f"Loaded merged panel: {PATHS['merged_panel']}", "INFO")
                else:
                    log("final_merged not available. Run Step 4 first.", "ERROR")
                    print("[ERROR] final_merged not available. Run Step 4 first.")
                    return

            final_merged = STATE["final_merged"]
            validate_final_merged(final_merged)

            # Get scenarios to run
            scenarios, order, all_sc, all_order = get_scenarios_to_run_from_ui()
            totalN = len(order)

            progress_bar.max = totalN + 1
            progress_bar.value = 0
            update_status("Starting run...", stage="init", current_idx=-1, total=totalN, stage_i=0)
            log(f"Scenario order (run=true): {order}", "INFO")

            results = {}
            facility_level = {}
            sd_data = {}

            for i, sid in enumerate(order):
                check_cancel("(before scenario)")

                sc = scenarios[sid]
                scen_start = time.time()

                progress_bar.value = i
                stage_bar.value = 0

                # Stage 1: prepare
                update_status(f"Scenario {i+1}/{totalN}: {sid} — preparing data", stage="prepare", current_idx=i, total=totalN, stage_i=1)
                para, tech_list, info, prep_stats = prepare_math_data_fast(
                    final_merged,
                    delta_ei_factor=sc["delta_ei_factor"],
                    baseline_ei_factor=sc["baseline_ei_factor"],
                    filter_negative_cost=sc["filter_negative_cost"]
                )
                log(f"[{sid}] Prep: facilities={prep_stats['facilities']:,} | kept_options={prep_stats['options_kept_total']:,} "
                    f"| dropped_nan={prep_stats['dropped_nan']:,} | dropped_nonpos_delta={prep_stats['dropped_nonpos_delta']:,} "
                    f"| dropped_nonpos_cost={prep_stats['dropped_nonpos_cost']:,}", "INFO")

                check_cancel("(after prepare)")

                # Stage 2: equilibrium
                update_status(f"Scenario {i+1}/{totalN}: {sid} — searching equilibrium price", stage="price_search", current_idx=i, total=totalN, stage_i=2)

                def _status_cb(txt):
                    progress_html.value = f"<b>Status:</b> {txt} <span style='color:#555'>(stage: price_search)</span>"

                pc_eq = find_equilibrium_price(
                    para, tech_list,
                    floor=sc["price_floor"], ceiling=sc["price_ceiling"],
                    price_min=float(price_min.value),
                    price_cap=float(price_cap.value),
                    price_step=float(price_step.value),
                    cancel_cb=lambda w: check_cancel(w),
                    status_cb=_status_cb
                )
                log(f"[{sid}] Equilibrium price = {pc_eq:.4f}", "INFO")

                # Stage 3: outcomes (facility table once)
                update_status(f"Scenario {i+1}/{totalN}: {sid} — computing outcomes", stage="main", current_idx=i, total=totalN, stage_i=3)
                imb, facility_tbl, td, ts = build_facility_table(para, tech_list, info, pc_eq)
                facility_tbl["scenario"] = sid
                facility_level[sid] = facility_tbl.copy()

                # Stage 4: SD curve
                update_status(f"Scenario {i+1}/{totalN}: {sid} — generating SD curve", stage="sd_curve", current_idx=i, total=totalN, stage_i=4)

                def _sd_progress(k, n, pc):
                    progress_html.value = f"<b>Status:</b> SD curve {k:,}/{n:,} — pc={pc:.2f} <span style='color:#555'>(stage: sd_curve)</span>"
                    timer_html.value = f"<b>Elapsed:</b> {fmt_elapsed(time.time() - start_time)}"

                sd_data[sid] = analyze_sd_curve_fast(
                    para, tech_list,
                    price_min=float(price_min.value),
                    price_cap=float(price_cap.value),
                    sd_step=float(sd_step.value),
                    cancel_cb=lambda w: check_cancel(w),
                    progress_cb=_sd_progress
                )

                # Stage 5: subsector
                update_status(f"Scenario {i+1}/{totalN}: {sid} — subsector aggregation", stage="subsector", current_idx=i, total=totalN, stage_i=5)
                sub = subsector_analysis(facility_tbl)

                total_base = float(facility_tbl["Emis_baseline"].sum())
                total_ets  = float(facility_tbl["Emis"].sum())
                red_pct = ((total_base - total_ets) / total_base) * 100 if total_base > 0 else 0.0

                results[sid] = {
                    "scenario_name": sc["name"],
                    "description": sc["description"],
                    "equilibrium_price": float(pc_eq),
                    "total_demand": float(td),
                    "total_supply": float(ts),
                    "imbalance": float(imb),
                    "emission_baseline": total_base,
                    "emission_with_ETS": total_ets,
                    "emission_reduction_pct": float(red_pct),
                    "subsector_results": sub
                }

                scenario_times.append(time.time() - scen_start)
                progress_bar.value = i + 1
                update_status(f"Scenario {i+1}/{totalN}: {sid} — completed", stage="done", current_idx=i, total=totalN, stage_i=6)
                log(f"[{sid}] Done. Emission reduction={red_pct:.2f}% | Imbalance={imb:.6g}", "INFO")

            check_cancel("(before summary/export)")

            # Summary + pivots
            update_status("Preparing summary + pivots...", stage="pivots", current_idx=totalN-1, total=totalN, stage_i=6)
            log("Preparing summary and pivot tables...", "INFO")

            summary_df = pd.DataFrame([{
                "Scenario": sid,
                "Scenario_Name": results[sid]["scenario_name"],
                "Description": results[sid]["description"],
                "Equilibrium_Price": round(results[sid]["equilibrium_price"], 4),
                "Total_Demand": results[sid]["total_demand"],
                "Total_Supply": results[sid]["total_supply"],
                "Market_Imbalance": results[sid]["imbalance"],
                "Emission_Baseline": results[sid]["emission_baseline"],
                "Emission_with_ETS": results[sid]["emission_with_ETS"],
                "Emission_Reduction_Pct": results[sid]["emission_reduction_pct"]
            } for sid in order])

            all_facilitys = pd.concat(facility_level.values(), ignore_index=True)

            unit_cost_rows, ratio_rows, emission_rows, intensity_rows, production_rows = [], [], [], [], []
            for sid in order:
                sub = results[sid]["subsector_results"]
                for _, row in sub.iterrows():
                    unit_cost_rows.append({
                        "Scenario": sid,
                        "Subsector": row["subsector"],
                        "Avg_Unit_Total_Cost_Change": row["avg_unit_total_cost_change"],
                        "Avg_Unit_Tech_Cost": row["avg_unit_tech_cost"],
                        "Avg_Unit_Carbon_Cost": row["avg_unit_carbon_cost"],
                    })
                    ratio_rows.append({"Scenario": sid, "Subsector": row["subsector"], "Unit_Cost_Change_Ratio": row["unit_cost_change_ratio"]})
                    emission_rows.append({"Scenario": sid, "Subsector": row["subsector"], "Emission_Reduction_Pct": row["emission_reduction_pct"]})
                    intensity_rows.append({"Scenario": sid, "Subsector": row["subsector"], "Carbon_Intensity_Change_Pct": row["carbon_intensity_change_pct"]})
                    production_rows.append({"Scenario": sid, "Subsector": row["subsector"], "Production_Change_Pct": row["production_change_pct"]})

            unit_cost_df = pd.DataFrame(unit_cost_rows)

            subsector_total_cost_pivot = unit_cost_df.pivot_table(index="Subsector", columns="Scenario", values="Avg_Unit_Total_Cost_Change", aggfunc="first")
            subsector_tech_cost_pivot  = unit_cost_df.pivot_table(index="Subsector", columns="Scenario", values="Avg_Unit_Tech_Cost", aggfunc="first")
            subsector_carbon_cost_pivot= unit_cost_df.pivot_table(index="Subsector", columns="Scenario", values="Avg_Unit_Carbon_Cost", aggfunc="first")

            subsector_unit_cost_ratio_pivot = pd.DataFrame(ratio_rows).pivot_table(index="Subsector", columns="Scenario", values="Unit_Cost_Change_Ratio", aggfunc="first")
            subsector_emission_pivot        = pd.DataFrame(emission_rows).pivot_table(index="Subsector", columns="Scenario", values="Emission_Reduction_Pct", aggfunc="first")
            subsector_intensity_pivot       = pd.DataFrame(intensity_rows).pivot_table(index="Subsector", columns="Scenario", values="Carbon_Intensity_Change_Pct", aggfunc="first")
            subsector_production_pivot      = pd.DataFrame(production_rows).pivot_table(index="Subsector", columns="Scenario", values="Production_Change_Pct", aggfunc="first")

            # Run summary card (pre-export)
            top = summary_df[["Scenario","Equilibrium_Price","Emission_Reduction_Pct","Market_Imbalance"]].copy()
            top["Equilibrium_Price"] = top["Equilibrium_Price"].map(lambda x: f"{x:.2f}")
            top["Emission_Reduction_Pct"] = top["Emission_Reduction_Pct"].map(lambda x: f"{x:.2f}%")
            top["Market_Imbalance"] = top["Market_Imbalance"].map(lambda x: f"{x:.4g}")
            summary_card.value = (
                "<div style='border:1px solid #ddd; border-radius:10px; padding:12px; background:#fafafa;'>"
                "<div style='font-weight:900; margin-bottom:6px;'>Run summary (pre-export)</div>"
                f"<div style='color:#555; font-size:12px; margin-bottom:8px;'>"
                f"Scenarios run: {len(order)} | Price bounds: [{float(price_min.value):.2f}, {float(price_cap.value):.2f}] "
                f"| Eq step: {float(price_step.value):g} | SD step: {float(sd_step.value):g}</div>"
                f"{top.to_html(index=False)}"
                "</div>"
            )

            check_cancel("(before export)")

            # Export
            update_status("Exporting Excel workbook...", stage="export", current_idx=totalN-1, total=totalN, stage_i=6)
            progress_bar.bar_style = "warning"
            stage_bar.bar_style = "warning"
            log("Exporting workbook...", "INFO")

            output_path = PATHS["scenario_results"]
            used_sheet_names = set()

            # Metadata sheet
            sc_all, order_all = get_all_scenarios_from_ui()
            metadata = {
                "run_timestamp_local": datetime.now().isoformat(sep=" ", timespec="seconds"),
                "step5_version": STEP5_VERSION,
                "merged_panel_path": str(PATHS.get("merged_panel")),
                "scenario_results_path": str(PATHS.get("scenario_results")),
                "price_min": float(price_min.value),
                "price_cap": float(price_cap.value),
                "price_step": float(price_step.value),
                "sd_step": float(sd_step.value),
                "scenarios_config_json": json.dumps({"order": order_all, "scenarios": sc_all}, indent=2),
            }
            metadata_df = pd.DataFrame(list(metadata.items()), columns=["key","value"])

            with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
                metadata_df.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["metadata"], used_sheet_names), index=False)
                summary_df.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["summary"], used_sheet_names), index=False)

                subsector_total_cost_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_total"], used_sheet_names))
                subsector_tech_cost_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_tech"], used_sheet_names))
                subsector_carbon_cost_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_carb"], used_sheet_names))
                subsector_unit_cost_ratio_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_ratio"], used_sheet_names))

                subsector_emission_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_emis"], used_sheet_names))
                subsector_intensity_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_int"], used_sheet_names))
                subsector_production_pivot.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["sub_prod"], used_sheet_names))

                all_facilitys.to_excel(writer, sheet_name=safe_sheet_name(SHEET_NAMES["facility"], used_sheet_names), index=False)

                for sid in order:
                    sd_sheet  = safe_sheet_name(f"SD_Curve_{sid}", used_sheet_names)
                    sub_sheet = safe_sheet_name(f"Subsector_Detail_{sid}", used_sheet_names)
                    sd_data[sid].to_excel(writer, sheet_name=sd_sheet, index=False)
                    results[sid]["subsector_results"].to_excel(writer, sheet_name=sub_sheet, index=False)

            progress_bar.value = totalN + 1
            progress_bar.bar_style = "success"
            stage_bar.bar_style = "success"
            update_status("Done ✅ Results exported.", stage="complete", current_idx=totalN-1, total=totalN, stage_i=6)
            log(f"Exported workbook: {output_path}", "INFO")
            print(f"\n[OK] Exported FULL workbook: {output_path}")

            if "refresh_file_status" in globals() and callable(globals()["refresh_file_status"]):
                try:
                    refresh_file_status()
                except Exception as e:
                    log(f"refresh_file_status() failed: {e}", "WARN")

        except KeyboardInterrupt:
            print("[INFO] Run cancelled.")
        except Exception as e:
            progress_bar.bar_style = "danger"
            stage_bar.bar_style = "danger"
            update_status(f"Error: {e}", stage="error", stage_i=0)
            log(f"Run failed: {e}", "ERROR")
            print(f"[ERROR] {e}")
        finally:
            btn_run_model.disabled = False
            btn_run_model.description = "Run scenarios + export results workbook"
            btn_run_model.button_style = "success"

            btn_cancel.disabled = True
            CANCEL["stop"] = False
            set_scenario_ui_enabled(True)
            update_validation()

btn_run_model.on_click(run_model)

# -----------------------------
# Build the Step 5 UI (WIZARD-SAFE: no Tabs)
# -----------------------------
load_default_scenarios()

helper_scenarios = widgets.HTML(
    "<div style='color:#555; font-size:12px; line-height:1.35;'>"
    "<b>Field meaning:</b> <code>delta_ei_factor</code> scales abatement potential (<code>delta_ei</code>), "
    "<code>baseline_ei_factor</code> scales baseline EI (<code>BASELINE_EI</code>). "
    "Floor/Ceiling constrain the equilibrium search bounds (in $/tCO₂). "
    "<br><b>Tip:</b> Use ⧉ to duplicate a row quickly.</div>"
)

def _hdr(txt, w):
    return widgets.HTML(f"<b>{txt}</b>", layout=widgets.Layout(width=w))

# Header row that matches the exact column widths used in make_scenario_row()
scenario_header = widgets.HBox(
    [
        _hdr("del",  "28px"),   # sel_del checkbox width
        _hdr("dup",  "34px"),   # duplicate button width
        _hdr("run",  "60px"),   # run checkbox width
        _hdr("ID",   "110px"),
        _hdr("Name", "260px"),
        _hdr("Description", "340px"),
        _hdr("delta", "120px"),
        _hdr("base",  "120px"),
        _hdr("floor", "70px"),
        _hdr("val",   "100px"),
        _hdr("ceiling", "85px"),
        _hdr("val",     "100px"),
        _hdr("exclude", "220px"),
    ],
    layout=widgets.Layout(
        flex_flow="row wrap",
        gap="8px",
        width="100%",
        align_items="center"
    )
)

scenario_header.layout = widgets.Layout(
    flex_flow="row wrap",
    gap="8px",
    width="100%",
    align_items="center",
    padding="6px 4px",
    border="1px solid #e0e0e0",
    background_color="#fafafa",
    border_radius="6px"
)

scenario_panel = widgets.VBox([
    widgets.HBox([btn_reset_defaults, btn_add, btn_delete, btn_run_all, btn_run_none],
                 layout=widgets.Layout(flex_flow="row wrap", gap="8px", width="98%")),
    helper_scenarios,
    validation_html,
    scenario_header,
    scenario_rows_box,
    widgets.HTML("<b>Save/Load scenario config</b>"),
    widgets.HBox([cfg_name, btn_save_cfg, btn_load_cfg], layout=widgets.Layout(flex_flow="row wrap", gap="8px", width="98%")),
    cfg_out
])

run_panel = widgets.VBox([
    widgets.HTML("<b>Run controls</b>"),
    widgets.HBox([price_step, price_min, price_cap, sd_step],
                 layout=widgets.Layout(flex_flow="row wrap", gap="12px", width="98%")),
    helper_run,
    progress_bar,
    stage_bar,
    timer_html,
    eta_html,
    progress_html,
    widgets.HBox([btn_run_model, btn_cancel], layout=widgets.Layout(gap="10px")),
    summary_card,
    log_pane,
    step5_out
])

header_html = widgets.HTML(
    f"<h3>Step 5 — Scenario Table + Run</h3>"
    f"<div style='color:#777; font-size:12px;'>Version: {STEP5_VERSION}</div>"
)

step5_box = widgets.VBox([
    header_html,
    widgets.HTML("<h4 style='margin:10px 0 6px 0;'>Scenarios</h4>"),
    scenario_panel,
    widgets.HTML("<hr style='margin:14px 0;'>"),
    widgets.HTML("<h4 style='margin:10px 0 6px 0;'>Run + Output</h4>"),
    run_panel
])

# Register for Step 7 wizard
globals()["step5_box"] = step5_box
globals().setdefault("WIZARD_STEPS", {})["s5"] = step5_box

# Initial validation state
update_validation()

# Display now (optional; wizard will also display it)
# display(step5_box)


# step 6

## STEP 6 full

In [ ]:

import pandas as pd
import numpy as np

if "_get_detail_df" not in globals():
    def _get_detail_df(xls, path, facility_df, scenario_id, level_choice, cache_read):
        """
        Returns: (detail_df, label_col, level_name)
        Uses facility table if available; otherwise uses Subsector_Detail_{scenario} if present.
        """
        level_choice = str(level_choice).lower().strip()
        sub_sheet = f"Subsector_Detail_{scenario_id}"

        # ---- ALL ----
        if level_choice == "all":
            if facility_df is not None:
                fsub = facility_df[facility_df["scenario"].astype(str).str.strip() == str(scenario_id).strip()].copy()
                return _detail_all_from_facility(fsub), "all", "All facilities"

            if sub_sheet in xls.sheet_names:
                df = cache_read(sub_sheet).copy()

                def _num(c):
                    return pd.to_numeric(df.get(c), errors="coerce").fillna(0.0)

                pb = float(_num("production_baseline").sum())
                pe = float(_num("production_with_ETS").sum())
                eb = float(_num("emission_baseline").sum())
                ee = float(_num("emission_with_ETS").sum())

                out = pd.DataFrame([{
                    "all": "All facilities",
                    "production_baseline": pb,
                    "production_with_ETS": pe,
                    "emission_baseline": eb,
                    "emission_with_ETS": ee,
                    "production_change_pct": ((pe - pb) / pb * 100) if pb > 0 else 0.0,
                    "carbon_intensity_baseline": (eb / pb) if pb > 0 else 0.0,
                    "carbon_intensity_with_ETS": (ee / pe) if pe > 0 else 0.0,
                    "carbon_intensity_change_pct": (((ee/pe)-(eb/pb))/(eb/pb)*100) if (pb>0 and pe>0 and (eb/pb)>0) else 0.0,
                    "emission_reduction_pct": ((eb - ee) / eb * 100) if eb > 0 else 0.0,
                }])
                return out, "all", "All facilities"

            raise ValueError("Aggregate requested but neither facility table nor Subsector_Detail sheet exists.")

        # ---- SUBSECTOR ----
        if level_choice == "subsector":
            if sub_sheet in xls.sheet_names:
                df = cache_read(sub_sheet).copy()
                if "subsector" in df.columns:
                    return df, "subsector", "Subsector"
                # tolerate common naming
                if "Subsector" in df.columns:
                    df = df.rename(columns={"Subsector": "subsector"})
                    return df, "subsector", "Subsector"

            if facility_df is None:
                raise ValueError("Subsector requested but facility table is missing and no Subsector_Detail sheet.")
            fsub = facility_df[facility_df["scenario"].astype(str).str.strip() == str(scenario_id).strip()].copy()
            if "subsector" not in fsub.columns:
                raise ValueError("Facility table missing 'subsector' column.")
            return _detail_from_facility(fsub, "subsector"), "subsector", "Subsector"

        # ---- SECTOR (default) ----
        if facility_df is None:
            raise ValueError("Sector requested but facility table missing.")
        fsub = facility_df[facility_df["scenario"].astype(str).str.strip() == str(scenario_id).strip()].copy()
        return _detail_from_facility(fsub, "sector"), "sector", "Sector"


In [ ]:
# ======================================================
# STEP 6 — FINAL
#
# Fixes + guarantees:
# ✅ Prevents Colab “random crash” by NOT auto-rendering huge galleries after plot generation
# ✅ Gallery shows only LAST N images (default 20)
# ✅ Disables Load/Generate/Zip while running (prevents double-click / re-entrancy)
# ✅ Leak catcher prevents plots/images leaking outside widget outputs
# ✅ Robust scenario loading (preferred 7 + all extras)
# ✅ Robust facility sheet detection (facility / Facility / facilty / Firm legacy)
# ✅ Alphabetical ordering in all categorical plots (case-insensitive)
# ✅ Bubble ΔCost is % of price; legend only 3 classes; Seller always green
# ✅ Prevent duplicate btn_launch_dashboard click handlers if cell rerun
#
# Requires: PATHS dict exists (from earlier steps), with PATHS["scenario_results"] optional.
# ======================================================

# Install calamine (safe pattern)
try:
    import python_calamine  # noqa
except Exception:
    !pip -q install python-calamine

import shutil
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, Image
import time
import functools
import hashlib
import pickle
import warnings
warnings.filterwarnings("ignore")

plt.ioff()  # prevents interactive auto-display in notebook output

# --------- Enhanced CSS ----------
display(widgets.HTML("""
<style>
.widget-label { white-space: normal !important; }
.widget-html { white-space: normal !important; }
.status-wrap { white-space: normal !important; word-break: break-word; }
.widget-select-multiple select { white-space: pre !important; }

.dashboard-title {
    color: #2c3e50;
    border-bottom: 2px solid #3498db;
    padding-bottom: 8px;
    margin-bottom: 15px;
}
.status-badge {
    display: inline-block;
    padding: 2px 8px;
    border-radius: 12px;
    font-size: 11px;
    font-weight: bold;
    margin-left: 10px;
}
.success { background: #d4edda; color: #155724; }
.warning { background: #fff3cd; color: #856404; }
.error   { background: #f8d7da; color: #721c24; }
.info    { background: #d1ecf1; color: #0c5460; }

.progress-label { font-size: 12px; color: #666; margin-top: 5px; }
.plot-title { font-weight: bold; color: #2c3e50; margin: 5px 0; font-size: 14px; }
</style>
"""))

# ============================================================
# Globals
# ============================================================
PLOT_DPI = 120
MAX_GALLERY_SHOW = 20 # show only last N images to prevent UI freezes

# ============================================================
# Cache
# ============================================================
class SmartCache:
    def __init__(self, max_size_mb=200):
        self.cache = {}
        self.max_size = max_size_mb * 1024 * 1024
        self.current_size = 0
        self.hits = 0
        self.misses = 0

    def _get_key(self, func_name, *args, **kwargs):
        key_parts = [func_name] + [str(a) for a in args] + [f"{k}={v}" for k, v in sorted(kwargs.items())]
        return hashlib.md5("|".join(key_parts).encode()).hexdigest()

    def _get_size(self, obj):
        return len(pickle.dumps(obj))

    def get(self, func_name, *args, **kwargs):
        key = self._get_key(func_name, *args, **kwargs)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None

    def set(self, func_name, result, *args, **kwargs):
        key = self._get_key(func_name, *args, **kwargs)
        size = self._get_size(result)
        if size + self.current_size > self.max_size:
            self.clear()
        self.cache[key] = result
        self.current_size += size

    def clear(self):
        self.cache.clear()
        self.current_size = 0
        self.hits = 0
        self.misses = 0
        gc.collect()

    def stats(self):
        total = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0
        return {
            "size_mb": self.current_size / (1024 * 1024),
            "items": len(self.cache),
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{hit_rate:.1%}",
            "total": total
        }

ENHANCED_CACHE = SmartCache(max_size_mb=200)

# ============================================================
# Core utilities
# ============================================================
def _open_excel(path: Path):
    try:
        return pd.ExcelFile(path, engine="calamine")
    except Exception:
        return pd.ExcelFile(path, engine="openpyxl")

def optimized_read_excel(path, sheet_name, use_cache=True):
    if use_cache:
        cached = ENHANCED_CACHE.get("read_excel", str(path), sheet_name)
        if cached is not None:
            return cached
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine="calamine")
    except Exception:
        df = pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    if use_cache:
        ENHANCED_CACHE.set("read_excel", df, str(path), sheet_name)
    return df

def _clear_pngs(outdir: Path):
    outdir.mkdir(parents=True, exist_ok=True)
    for p in outdir.glob("*.png"):
        try:
            p.unlink()
        except Exception:
            pass

def _find_facility_sheet(sheets):
    candidates = [
        "facility_Level_All_Data",
        "Facility_Level_All_Data",
        "facilty_Level_All_Data",   # common typo
        "Firm_Level_All_Data",
        "firm_Level_All_Data",
    ]
    sset = set(sheets)
    for c in candidates:
        if c in sset:
            return c
    return None

def _available_scenarios_from_excel(xls: pd.ExcelFile, path: Path):
    sheets = xls.sheet_names
    preferred = ["A","A_pf5","A_pf10","B","C","C_pc15","C_pc20"]

    def order_with_preferred(vals):
        vals = [str(v).strip() for v in vals if v is not None and str(v).strip() and str(v).lower() != "nan"]
        vals_set = set(vals)
        ordered = [s for s in preferred if s in vals_set]
        extras = sorted([s for s in vals if s not in set(preferred)])
        return ordered + extras

    if "Summary" in sheets:
        s = optimized_read_excel(path, "Summary")
        cols = {str(c).lower(): c for c in s.columns}
        scen_col = cols.get("scenario", None) or next((c for c in s.columns if "scenario" in str(c).lower()), None)
        if scen_col is not None:
            vals = s[scen_col].astype(str).str.strip().unique().tolist()
            if vals:
                return order_with_preferred(vals)

    fac_sheet = _find_facility_sheet(sheets)
    if fac_sheet:
        f = optimized_read_excel(path, fac_sheet)
        cols = {str(c).lower(): c for c in f.columns}
        scen_col = cols.get("scenario", None)
        if scen_col is not None:
            vals = f[scen_col].astype(str).str.strip().unique().tolist()
            if vals:
                return order_with_preferred(vals)

    scenarios = set()
    for sname in sheets:
        if sname.startswith("Subsector_Detail_"):
            scenarios.add(sname.replace("Subsector_Detail_", "", 1))
        if sname.startswith("SD_Curve_"):
            scenarios.add(sname.replace("SD_Curve_", "", 1))
    return order_with_preferred(sorted(scenarios))

def _get_eq_price(summary_df: pd.DataFrame, scenario_id: str):
    if summary_df is None or summary_df.empty:
        return None
    cols = {str(c).lower(): c for c in summary_df.columns}
    scen_col = cols.get("scenario", None) or next((c for c in summary_df.columns if "scenario" in str(c).lower()), None)
    pc_col = cols.get("equilibrium_price", None) or next((c for c in summary_df.columns if "equilibrium" in str(c).lower()), None)
    if scen_col is None or pc_col is None:
        return None
    row = summary_df.loc[summary_df[scen_col].astype(str).str.strip() == str(scenario_id).strip()]
    if row.empty:
        return None
    try:
        return float(row.iloc[0][pc_col])
    except Exception:
        return None

def _get_summary_row(summary_df: pd.DataFrame, scenario_id: str):
    if summary_df is None or summary_df.empty:
        return {}
    cols = {str(c).lower(): c for c in summary_df.columns}
    scen_col = cols.get("scenario", None) or next((c for c in summary_df.columns if "scenario" in str(c).lower()), None)
    if scen_col is None:
        return {}
    row = summary_df.loc[summary_df[scen_col].astype(str).str.strip() == str(scenario_id).strip()]
    if row.empty:
        return {}
    return row.iloc[0].to_dict()

def fixed_save_plot(fig, path: Path, dpi=None):
    global PLOT_DPI
    if dpi is None:
        dpi = PLOT_DPI
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(path, dpi=int(dpi), bbox_inches="tight")
    plt.close(fig)

def fixed_plot_generation(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        plt.close("all")
        try:
            return func(*args, **kwargs)
        finally:
            plt.close("all")
            gc.collect()
    return wrapper

def _figsize_for_categories(n, base_w=7, per_cat=0.25, min_h=5, max_h=24, per_cat_w=0.0):
    n = max(int(n), 1)
    w = base_w + per_cat_w * n
    h = max(min_h, per_cat * n)
    return (w, min(h, max_h))

# ============================================================
# Detail tables
# ============================================================
def _detail_from_facility(df_facility_scenario: pd.DataFrame, group_col: str):
    df = df_facility_scenario.copy()

    required_cols = ["q0","qst","Emis_baseline","Emis","del_u","py","unit_tech_cost","unit_carbon_cost_change","N"]
    for col in required_cols:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    if group_col not in df.columns:
        raise ValueError(f"Facility table missing '{group_col}'")

    df[group_col] = df[group_col].astype(str).str.strip()
    df = df.dropna(subset=[group_col]).copy()

    out = df.groupby(group_col, as_index=False).agg(
        num_facilities=("del_u", "size"),
        production_baseline=("q0", "sum"),
        production_with_ETS=("qst", "sum"),
        emission_baseline=("Emis_baseline", "sum"),
        emission_with_ETS=("Emis", "sum"),
    )

    out["production_change_pct"] = np.where(
        out["production_baseline"] > 0,
        (out["production_with_ETS"] - out["production_baseline"]) / out["production_baseline"] * 100,
        0
    )

    out["carbon_intensity_baseline"] = np.where(
        out["production_baseline"] > 0, out["emission_baseline"] / out["production_baseline"], 0
    )
    out["carbon_intensity_with_ETS"] = np.where(
        out["production_with_ETS"] > 0, out["emission_with_ETS"] / out["production_with_ETS"], 0
    )
    out["carbon_intensity_change_pct"] = np.where(
        out["carbon_intensity_baseline"] > 0,
        (out["carbon_intensity_with_ETS"] - out["carbon_intensity_baseline"]) / out["carbon_intensity_baseline"] * 100,
        0
    )

    # weighted averages (weights=qst)
    df["w_num"] = df["del_u"] * df["qst"]
    df["w_den"] = df["qst"]
    tmp = df.groupby(group_col, as_index=False)[["w_num","w_den"]].sum()
    tmp["avg_unit_total_cost_change"] = np.where(tmp["w_den"] > 0, tmp["w_num"] / tmp["w_den"], 0)
    out = out.merge(tmp[[group_col,"avg_unit_total_cost_change"]], on=group_col, how="left")

    df["p_num"] = df["py"] * df["qst"]
    df["p_den"] = df["qst"]
    ptmp = df.groupby(group_col, as_index=False)[["p_num","p_den"]].sum()
    ptmp["avg_product_price"] = np.where(ptmp["p_den"] > 0, ptmp["p_num"] / ptmp["p_den"], np.nan)
    out = out.merge(ptmp[[group_col,"avg_product_price"]], on=group_col, how="left")
    out["unit_cost_change_ratio"] = np.where(
        out["avg_product_price"] > 0,
        (out["avg_unit_total_cost_change"] / out["avg_product_price"]) * 100,
        0
    )

    df["t_num"] = df["unit_tech_cost"] * df["qst"]
    ttmp = df.groupby(group_col, as_index=False)[["t_num","w_den"]].sum()
    ttmp["avg_unit_tech_cost"] = np.where(ttmp["w_den"] > 0, ttmp["t_num"] / ttmp["w_den"], np.nan)
    out = out.merge(ttmp[[group_col,"avg_unit_tech_cost"]], on=group_col, how="left")

    df["c_num"] = df["unit_carbon_cost_change"] * df["qst"]
    ctmp = df.groupby(group_col, as_index=False)[["c_num","w_den"]].sum()
    ctmp["avg_unit_carbon_cost"] = np.where(ctmp["w_den"] > 0, ctmp["c_num"] / ctmp["w_den"], np.nan)
    out = out.merge(ctmp[[group_col,"avg_unit_carbon_cost"]], on=group_col, how="left")

    out["emission_reduction_pct"] = np.where(
        out["emission_baseline"] > 0,
        (out["emission_baseline"] - out["emission_with_ETS"]) / out["emission_baseline"] * 100,
        0
    )
    return out

def _detail_all_from_facility(df_facility_scenario: pd.DataFrame):
    df = df_facility_scenario.copy()
    df["__all__"] = "All facilities"
    out = _detail_from_facility(df, "__all__")
    out = out.rename(columns={"__all__":"all"})
    return out

def _get_detail_df(xls, path, facility_df, scenario_id, level_choice, cache_read):
    level_choice = str(level_choice).lower().strip()
    sub_sheet = f"Subsector_Detail_{scenario_id}"

    if level_choice == "all":
        if facility_df is not None:
            fsub = facility_df[facility_df["scenario"].astype(str).str.strip()==str(scenario_id).strip()].copy()
            return _detail_all_from_facility(fsub), "all", "All facilities"
        if sub_sheet in xls.sheet_names:
            df = cache_read(sub_sheet).copy()
            def num(x): return pd.to_numeric(df.get(x), errors="coerce")
            pb = num("production_baseline").sum()
            pe = num("production_with_ETS").sum()
            eb = num("emission_baseline").sum()
            ee = num("emission_with_ETS").sum()
            out = pd.DataFrame([{
                "all": "All facilities",
                "production_baseline": pb,
                "production_with_ETS": pe,
                "emission_baseline": eb,
                "emission_with_ETS": ee,
                "production_change_pct": ((pe - pb) / pb * 100) if pb > 0 else 0,
                "carbon_intensity_baseline": (eb / pb) if pb > 0 else 0,
                "carbon_intensity_with_ETS": (ee / pe) if pe > 0 else 0,
                "carbon_intensity_change_pct": (((ee/pe)-(eb/pb))/(eb/pb)*100) if (pb>0 and pe>0 and (eb/pb)>0) else 0,
                "emission_reduction_pct": ((eb - ee) / eb * 100) if eb > 0 else 0,
            }])
            return out, "all", "All facilities"
        raise ValueError("Aggregate requested but no facility table and no Subsector_Detail sheet.")

    if sub_sheet in xls.sheet_names:
        df = cache_read(sub_sheet)
        if level_choice == "subsector" and "subsector" in df.columns:
            return df, "subsector", "Subsector"
        if level_choice == "sector":
            if facility_df is None:
                raise ValueError("Sector requested but facility table missing.")
            fsub = facility_df[facility_df["scenario"].astype(str).str.strip()==str(scenario_id).strip()].copy()
            return _detail_from_facility(fsub, "sector"), "sector", "Sector"

    if facility_df is None:
        raise ValueError("No detail sheet and facility table missing.")
    fsub = facility_df[facility_df["scenario"].astype(str).str.strip()==str(scenario_id).strip()].copy()
    grp = "subsector" if level_choice == "subsector" else "sector"
    df = _detail_from_facility(fsub, grp)
    return df, grp, grp.title()

# ============================================================
# Plot primitives
# ============================================================
def _bubble_sizes(size_raw: pd.Series):
    raw = size_raw.clip(lower=0).values.astype(float)
    p10, p90 = np.percentile(raw, [10, 90])
    p10 = max(p10, 1e-9)
    p90 = max(p90, p10 * 1.01)
    scaled = np.clip((raw - p10) / (p90 - p10), 0, 1)
    return 20 + 280 * scaled

def _bubble_prepare(facility_df: pd.DataFrame, scenario_id: str, size_by: str, sector_filter=None):
    d = facility_df.copy()
    d["scenario"] = d["scenario"].astype(str).str.strip()
    d = d[d["scenario"] == str(scenario_id).strip()].copy()
    d["sector"] = d["sector"].astype(str).str.strip()

    for c in ["q0","qst","del_u","N","output_change_pct","py"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    if sector_filter:
        d = d[d["sector"].astype(str).isin(set(sector_filter))].copy()

    if size_by not in ["q0","qst"]:
        size_by = "q0"
    d = d.dropna(subset=["sector", size_by]).copy()
    return d, size_by

def _sector_order_alpha(d: pd.DataFrame):
    return sorted(d["sector"].dropna().astype(str).str.strip().unique().tolist(), key=lambda x: str(x).lower())

@fixed_plot_generation
def _plot_barh(df, label_col, value_col, title, xlabel, outpath, color_func=None, fmt="{:.2f}%"):
    df = df.copy()
    df[label_col] = df[label_col].astype(str).str.strip()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[label_col, value_col]).copy()
    df = df.sort_values(label_col, key=lambda s: s.str.lower()).reset_index(drop=True)  # ✅ alpha

    labels = df[label_col].values
    values = df[value_col].values.astype(float)
    n = len(df)

    fig, ax = plt.subplots(figsize=_figsize_for_categories(n))
    if color_func:
        colors = [color_func(v) for v in values]
        ax.barh(range(n), values, color=colors, edgecolor="black", linewidth=1.2)
    else:
        ax.barh(range(n), values, edgecolor="black", linewidth=1.2)

    ax.set_yticks(range(n))
    ax.set_yticklabels(labels, fontsize=9, fontweight="bold")
    ax.axvline(0, linewidth=1.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)

    xmin = min(0.0, float(np.nanmin(values)) if n else 0.0)
    xmax = max(0.0, float(np.nanmax(values)) if n else 0.0)
    rng = max(xmax - xmin, 1e-6)
    pad = 0.10 * rng + 0.5
    ax.set_xlim(xmin - pad, xmax + pad)
    xL, xR = ax.get_xlim()

    for i, v in enumerate(values):
        if v >= 0:
            x = min(v + 0.02*rng, xR - 0.02*rng); ha="left"
        else:
            x = max(v - 0.02*rng, xL + 0.02*rng); ha="right"
        ax.text(x, i, fmt.format(v), va="center", ha=ha, fontsize=8, fontweight="bold")

    fixed_save_plot(fig, outpath)

@fixed_plot_generation
def _boxplot_by_group(df, group_col, value_col, title, xlabel, outpath, clip_outliers=False):
    df = df.copy()
    df[group_col] = df[group_col].astype(str).str.strip()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[group_col, value_col]).copy()

    if clip_outliers and len(df) > 10:
        lo, hi = df[value_col].quantile([0.01, 0.99]).values
        df[value_col] = df[value_col].clip(lo, hi)

    groups = sorted(df[group_col].unique(), key=lambda x: str(x).lower())
    data = [df.loc[df[group_col] == g, value_col].values for g in groups]

    fig, ax = plt.subplots(figsize=_figsize_for_categories(len(groups)))
    ax.boxplot(data, labels=groups, vert=False, showfliers=not clip_outliers)
    ax.axvline(0, linewidth=1.6)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel(xlabel, fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)
    fixed_save_plot(fig, outpath)

# ============================================================
# Plot families
# ============================================================
@fixed_plot_generation
def plot_impacts_by_level(detail_df, label_col, level_name, scenario_id, eq_price, outdir: Path, prefix="T1_"):
    df = detail_df.copy()
    df[label_col] = df[label_col].astype(str).str.strip()
    df = df.sort_values(label_col, key=lambda s: s.str.lower()).reset_index(drop=True)  # ✅ alpha
    price_txt = f"${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc N/A"

    def ci_color(x):
        if x < -10: return "darkgreen"
        if x < 0:   return "lightgreen"
        if x < 5:   return "lightcoral"
        return "darkred"

    def cost_color(x):
        if x < -1: return "darkblue"
        if x < 0:  return "steelblue"
        if x < 1:  return "salmon"
        return "darkred"

    def out_color(x):
        if x > 2:  return "darkgreen"
        if x > 0:  return "lightgreen"
        if x > -2: return "lightcoral"
        return "darkred"

    _plot_barh(
        df, label_col, "carbon_intensity_change_pct",
        title=f"Carbon Intensity Change by {level_name}\nScenario {scenario_id} at equilibrium Pc {price_txt}",
        xlabel="Carbon Intensity Change (%)",
        outpath=outdir / f"{prefix}Figure1_CI_Change_{level_name}_S{scenario_id}.png",
        color_func=ci_color, fmt="{:.2f}%"
    )

    _plot_barh(
        df, label_col, "unit_cost_change_ratio",
        title=f"Unit Cost Change (% of price) by {level_name}\nScenario {scenario_id} at equilibrium Pc {price_txt}",
        xlabel="Unit Cost Change (% of product price)",
        outpath=outdir / f"{prefix}Figure2_Cost_Ratio_{level_name}_S{scenario_id}.png",
        color_func=cost_color, fmt="{:.3f}%"
    )

    _plot_barh(
        df, label_col, "production_change_pct",
        title=f"Output Change by {level_name}\nScenario {scenario_id} at equilibrium Pc {price_txt}",
        xlabel="Output Change (%)",
        outpath=outdir / f"{prefix}Figure3_Output_Change_{level_name}_S{scenario_id}.png",
        color_func=out_color, fmt="{:.3f}%"
    )

@fixed_plot_generation
def plot_equilibrium_distributions(facility_df, scenario_id, eq_price, outdir: Path,
                                   neutral_band_pct=0.00, clip_outliers=False, sector_filter=None,
                                   prefix="T2_"):
    d = facility_df.copy()
    d = d[d["scenario"].astype(str).str.strip() == str(scenario_id).strip()].copy()
    if sector_filter:
        d = d[d["sector"].astype(str).isin(set(sector_filter))].copy()

    price_txt = f"${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc N/A"
    for c in ["del_u","output_change_pct","N"]:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    d["sector"] = d["sector"].astype(str).str.strip()
    d = d.dropna(subset=["sector","del_u","output_change_pct","N"]).copy()

    _boxplot_by_group(d, "sector", "del_u",
                      title=f"Unit ΔCost (del_u) by Sector\nScenario {scenario_id} at equilibrium Pc {price_txt}",
                      xlabel="Unit ΔCost (del_u)",
                      outpath=outdir / f"{prefix}Fig1_DeltaCost_Box_S{scenario_id}.png",
                      clip_outliers=clip_outliers)

    eps = float(neutral_band_pct)
    def _cat(x):
        if x < -eps: return "Negative"
        if x >  eps: return "Positive"
        return "Neutral"
    d["delta_output_sign"] = d["output_change_pct"].apply(_cat)

    counts = (d.groupby(["sector","delta_output_sign"]).size()
              .unstack(fill_value=0)
              .reindex(columns=["Negative","Neutral","Positive"], fill_value=0)
              .sort_index(key=lambda idx: idx.astype(str).str.lower()))

    fig, ax = plt.subplots(figsize=_figsize_for_categories(len(counts.index), base_w=11, per_cat=0.28))
    counts.plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(f"ΔOutput sign by Sector\nScenario {scenario_id} at equilibrium Pc {price_txt} (neutral ±{eps}%)", fontweight="bold")
    ax.set_xlabel("Sector", fontweight="bold")
    ax.set_ylabel("Number of facilities", fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    fixed_save_plot(fig, outdir / f"{prefix}Fig2_OutputSign_Count_S{scenario_id}.png")

    _boxplot_by_group(d, "sector", "output_change_pct",
                      title=f"ΔOutput (%) by Sector\nScenario {scenario_id} at equilibrium Pc {price_txt}",
                      xlabel="ΔOutput (%)",
                      outpath=outdir / f"{prefix}Fig3_DeltaOutput_Box_S{scenario_id}.png",
                      clip_outliers=clip_outliers)

    _boxplot_by_group(d, "sector", "N",
                      title=f"N distribution by Sector\nScenario {scenario_id} at equilibrium Pc {price_txt}",
                      xlabel="N (tCO₂). + buyer, − seller",
                      outpath=outdir / f"{prefix}Fig4_CarbonPositionN_Box_S{scenario_id}.png",
                      clip_outliers=clip_outliers)

@fixed_plot_generation
def plot_bubble_cost_by_sector(facility_df, scenario_id, eq_price, outdir: Path,
                               size_by="q0", order_mode="alpha", jitter=0.18, alpha=0.65,
                               neutral_n_band=1e-6, sector_filter=None,
                               prefix="B1_"):
    d, size_by = _bubble_prepare(facility_df, scenario_id, size_by, sector_filter=sector_filter)
    for col in ["del_u","N","py",size_by]:
        if col not in d.columns:
            return False

    d["del_u"] = pd.to_numeric(d["del_u"], errors="coerce")
    d["py"]    = pd.to_numeric(d["py"], errors="coerce")
    d["N"]     = pd.to_numeric(d["N"], errors="coerce")

    d["del_u_pct"] = np.where(d["py"] > 0, (d["del_u"] / d["py"]) * 100.0, np.nan)
    d = d.dropna(subset=["sector","del_u_pct","N",size_by]).copy()

    def pos_bucket(n):
        if n > neutral_n_band: return "Buyer (N>0)"
        if n < -neutral_n_band: return "Seller (N<0)"
        return "Neutral (~0)"
    d["position"] = d["N"].apply(pos_bucket)

    if order_mode == "median":
        order = d.groupby("sector")["del_u_pct"].median().sort_values(ascending=False).index.tolist()
    else:
        order = _sector_order_alpha(d)

    y_map = {s:i for i,s in enumerate(order)}
    d["y"] = d["sector"].map(y_map).astype(float)

    rng = np.random.default_rng(42)
    d["y_j"] = d["y"] + rng.uniform(-jitter, jitter, size=len(d))
    d["s"] = _bubble_sizes(d[size_by])

    color_map = {"Seller (N<0)":"green", "Buyer (N>0)":"orange", "Neutral (~0)":"gray"}

    fig, ax = plt.subplots(figsize=(12, max(5, 0.35*len(order))))
    for pos, sub in d.groupby("position"):
        ax.scatter(sub["del_u_pct"], sub["y_j"], s=sub["s"], alpha=alpha,
                   edgecolors="black", linewidths=0.35,
                   marker="o", color=color_map.get(pos,"gray"), label=pos)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([s.replace("_"," ") for s in order], fontweight="bold")
    ax.axvline(0, color="black", linewidth=2)
    price_txt = f"Pc=${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc=Equilibrium"
    ax.set_title(f"All facilities: ΔCost (% of price) by Sector — scenario_{scenario_id} ({price_txt})", fontweight="bold")
    ax.set_xlabel("Unit ΔCost as % of product price", fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)
    ax.legend(loc="best", frameon=True, fontsize=9)
    fixed_save_plot(fig, outdir / f"{prefix}Bubble_DeltaCostPct_bySector_{scenario_id}.png")
    return True

@fixed_plot_generation
def plot_bubble_n_by_sector(facility_df, scenario_id, eq_price, outdir: Path,
                            size_by="q0", order_mode="alpha", jitter=0.18, alpha=0.65,
                            neutral_n_band=1e-6, sector_filter=None, prefix="B2_"):
    d, size_by = _bubble_prepare(facility_df, scenario_id, size_by, sector_filter=sector_filter)
    if "N" not in d.columns:
        return False
    d["N"] = pd.to_numeric(d["N"], errors="coerce")
    d = d.dropna(subset=["sector","N",size_by]).copy()

    def pos_bucket(n):
        if n > neutral_n_band: return "Buyer (N>0)"
        if n < -neutral_n_band: return "Seller (N<0)"
        return "Neutral (~0)"
    d["position"] = d["N"].apply(pos_bucket)

    if order_mode == "median":
        order = d.groupby("sector")["N"].median().sort_values(ascending=False).index.tolist()
    else:
        order = _sector_order_alpha(d)

    y_map = {s:i for i,s in enumerate(order)}
    d["y"] = d["sector"].map(y_map).astype(float)
    rng = np.random.default_rng(42)
    d["y_j"] = d["y"] + rng.uniform(-jitter, jitter, size=len(d))
    d["s"] = _bubble_sizes(d[size_by])

    colors = {"Buyer (N>0)":"orange","Seller (N<0)":"green","Neutral (~0)":"gray"}
    fig, ax = plt.subplots(figsize=(12, max(5, 0.35*len(order))))
    for label, sub in d.groupby("position"):
        ax.scatter(sub["N"], sub["y_j"], s=sub["s"], alpha=alpha,
                   edgecolors="black", linewidths=0.35,
                   color=colors.get(label,"gray"), marker="o", label=label)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([s.replace("_"," ") for s in order], fontweight="bold")
    ax.axvline(0, color="black", linewidth=2)
    price_txt = f"Pc=${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc=Equilibrium"
    ax.set_title(f"All facilities: Carbon Position (N) by Sector — scenario_{scenario_id} ({price_txt})", fontweight="bold")
    ax.set_xlabel("N (tCO₂)  (+ buyer demand, − seller supply)", fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)
    ax.legend(loc="best", frameon=True)
    fixed_save_plot(fig, outdir / f"{prefix}Scatter_CarbonPositionN_bySector_{scenario_id}.png")
    return True

@fixed_plot_generation
def plot_bubble_output_sign(facility_df, scenario_id, eq_price, outdir: Path,
                            size_by="q0", order_mode="alpha", jitter=0.18, alpha=0.70,
                            neutral_band_pct=0.0, show_counts=False, counts_font=10,
                            sector_filter=None, prefix="B4_"):
    d, size_by = _bubble_prepare(facility_df, scenario_id, size_by, sector_filter=sector_filter)
    if "output_change_pct" not in d.columns:
        return False
    d["output_change_pct"] = pd.to_numeric(d["output_change_pct"], errors="coerce")
    d = d.dropna(subset=["sector","output_change_pct",size_by]).copy()

    eps = float(neutral_band_pct)
    def out_bucket(x):
        if x < -eps: return "Negative"
        if x > eps:  return "Positive"
        return "Neutral"
    d["out_sign"] = d["output_change_pct"].apply(out_bucket)

    if order_mode == "median":
        order = d.groupby("sector")["output_change_pct"].median().sort_values(ascending=False).index.tolist()
    else:
        order = _sector_order_alpha(d)

    y_map = {s:i for i,s in enumerate(order)}
    d["y"] = d["sector"].map(y_map).astype(float)
    rng = np.random.default_rng(42)
    d["y_j"] = d["y"] + rng.uniform(-jitter, jitter, size=len(d))
    d["s"] = _bubble_sizes(d[size_by])

    colors = {"Negative":"red","Neutral":"gray","Positive":"green"}
    fig, ax = plt.subplots(figsize=(12, max(5, 0.35*len(order))))
    for label, sub in d.groupby("out_sign"):
        ax.scatter(sub["output_change_pct"], sub["y_j"], s=sub["s"], alpha=alpha,
                   edgecolors="black", linewidths=0.35,
                   color=colors.get(label,"gray"), marker="o", label=label)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([s.replace("_"," ") for s in order], fontweight="bold")
    ax.axvline(0, color="black", linewidth=2)
    price_txt = f"Pc=${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc=Equilibrium"
    ax.set_title(f"All facilities: ΔOutput (%) by Sector — scenario_{scenario_id} ({price_txt})", fontweight="bold")
    ax.set_xlabel("ΔOutput (%)", fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)
    ax.legend(loc="best", frameon=True, title="ΔOutput sign")
    fixed_save_plot(fig, outdir / f"{prefix}Scatter_OutputSign_bySector_{scenario_id}.png")
    return True

def plot_bubble_pack(facility_df, scenario_id, eq_price, outdir: Path,
                     size_by="q0", order_mode="alpha", neutral_band_pct=0.0,
                     include_counts_plot=True, sector_filter=None):
    ok = True
    ok &= bool(plot_bubble_cost_by_sector(facility_df, scenario_id, eq_price, outdir,
                                         size_by=size_by, order_mode=order_mode, sector_filter=sector_filter, prefix="B1_"))
    ok &= bool(plot_bubble_n_by_sector(facility_df, scenario_id, eq_price, outdir,
                                      size_by=size_by, order_mode=order_mode, sector_filter=sector_filter, prefix="B2_"))
    ok &= bool(plot_bubble_output_sign(facility_df, scenario_id, eq_price, outdir,
                                       size_by=size_by, order_mode=order_mode, neutral_band_pct=neutral_band_pct,
                                       show_counts=False, sector_filter=sector_filter, prefix="B4_"))
    # (Counts plot disabled in this production-safe version to reduce render load; you can re-enable if needed.)
    return ok

@fixed_plot_generation
def plot_market_mechanics_price_on_y(cache_read, xls: pd.ExcelFile, scenario_id, eq_price, outdir: Path,
                                     label_equilibrium=True, prefix="T3_"):
    sheet = f"SD_Curve_{scenario_id}"
    if sheet not in xls.sheet_names:
        return False

    sd = cache_read(sheet)
    for c in ["carbon_price","total_demand","total_supply","imbalance"]:
        sd[c] = pd.to_numeric(sd[c], errors="coerce")
    sd = sd.dropna(subset=["carbon_price","total_demand","total_supply","imbalance"]).copy()
    sd = sd.sort_values("carbon_price")

    eq_row = None
    if eq_price is not None and len(sd) > 0:
        idx = (sd["carbon_price"] - float(eq_price)).abs().idxmin()
        eq_row = sd.loc[idx].to_dict()

    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot(sd["total_demand"], sd["carbon_price"], label="Total Demand", linewidth=2)
    ax.plot(sd["total_supply"], sd["carbon_price"], label="Total Supply", linewidth=2)

    if eq_row and label_equilibrium:
        ax.scatter([eq_row["total_demand"]], [eq_row["carbon_price"]], s=80, color="red", zorder=3)
        ax.annotate(f"Pc={eq_row['carbon_price']:.2f}",
                    xy=(eq_row["total_demand"], eq_row["carbon_price"]),
                    xytext=(8,8), textcoords="offset points",
                    fontsize=9, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.3", alpha=0.2))

    ax.set_title(f"SD Curve (Price on Y-axis) — {scenario_id}", fontweight="bold")
    ax.set_xlabel("Quantity (tCO₂)", fontweight="bold")
    ax.set_ylabel("Carbon price ($/tCO₂)", fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend()
    fixed_save_plot(fig, outdir / f"{prefix}Fig1_SD_Curve_PriceOnY_S{scenario_id}.png")

    fig, ax = plt.subplots(figsize=(8,5))
    ax.plot(sd["imbalance"], sd["carbon_price"], linewidth=2)
    ax.axvline(0, color="black", linewidth=1.6)

    if eq_row and label_equilibrium:
        ax.scatter([eq_row["imbalance"]], [eq_row["carbon_price"]], s=80, color="red", zorder=3)

    ax.set_title(f"Imbalance vs Price — {scenario_id}", fontweight="bold")
    ax.set_xlabel("Imbalance (Demand − Supply)", fontweight="bold")
    ax.set_ylabel("Carbon price ($/tCO₂)", fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.3)
    fixed_save_plot(fig, outdir / f"{prefix}Fig2_Imbalance_PriceOnY_S{scenario_id}.png")
    return True


@fixed_plot_generation
def plot_buyer_seller_structure(facility_df, scenario_id, eq_price, outdir: Path,
                                sector_filter=None, prefix="T4_",
                                sort_counts_by="alpha", sort_volume_by="alpha"):
    d = facility_df.copy()
    d = d[d["scenario"].astype(str).str.strip() == str(scenario_id).strip()].copy()
    if sector_filter:
        d = d[d["sector"].astype(str).isin(set(sector_filter))].copy()

    d["N"] = pd.to_numeric(d["N"], errors="coerce")
    d["sector"] = d["sector"].astype(str).str.strip()
    d = d.dropna(subset=["sector","N"]).copy()
    price_txt = f"${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc N/A"

    d["is_buyer"]  = (d["N"] > 0).astype(int)
    d["is_seller"] = (d["N"] < 0).astype(int)

    counts = d.groupby("sector")[["is_buyer","is_seller"]].sum()
    counts = counts.rename(columns={"is_buyer":"Buyers","is_seller":"Sellers"})
    counts["Total"] = counts["Buyers"] + counts["Sellers"]
    counts["BuyerShare"] = np.where(counts["Total"] > 0, counts["Buyers"] / counts["Total"], 0)

    counts = counts.sort_index(key=lambda idx: idx.astype(str).str.lower())

    fig_h = max(5, 0.30*len(counts))
    fig, ax = plt.subplots(figsize=(11, fig_h))
    # Consistent colors: sellers green, buyers orange
    ax.barh(counts.index, counts["Sellers"], label="Sellers", color="green", alpha=0.65)
    ax.barh(counts.index, counts["Buyers"], left=counts["Sellers"], label="Buyers", color="orange", alpha=0.65)
    ax.set_title(f"Buyer/Seller Counts by Sector\nScenario {scenario_id} at equilibrium Pc {price_txt}", fontweight="bold")
    ax.set_xlabel("Number of facilities", fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)
    ax.legend(loc="best", frameon=True)
    fixed_save_plot(fig, outdir / f"{prefix}Fig1_BuyerSeller_Counts_S{scenario_id}.png")

    demand = d.loc[d["N"] > 0].groupby("sector")["N"].sum()
    supply = -d.loc[d["N"] < 0].groupby("sector")["N"].sum()
    vol = pd.DataFrame({"Demand": demand, "Supply": supply}).fillna(0)
    vol["Net"] = vol["Demand"] - vol["Supply"]
    vol = vol.sort_index(key=lambda idx: idx.astype(str).str.lower())

    fig_h = max(5, 0.30*len(vol))
    fig, ax = plt.subplots(figsize=(11, fig_h))
    ax.barh(vol.index, -vol["Supply"], label="Supply (Sellers, -)", alpha=0.70, color="green")
    ax.barh(vol.index,  vol["Demand"], label="Demand (Buyers, +)", alpha=0.70, color="orange")
    ax.axvline(0, color="black", linewidth=1.8)
    ax.set_title(f"Carbon Volumes by Sector (D+ vs S-)\nScenario {scenario_id} at equilibrium Pc {price_txt}", fontweight="bold")
    ax.set_xlabel("tCO₂ (left=supply, right=demand)", fontweight="bold")
    ax.grid(True, axis="x", linestyle="--", alpha=0.3)
    ax.legend(loc="best", frameon=True)
    fixed_save_plot(fig, outdir / f"{prefix}Fig2_DemandSupply_Diverging_S{scenario_id}.png")
    return True

@fixed_plot_generation
def plot_cost_decomposition(detail_df, label_col, level_name, scenario_id, eq_price, outdir: Path, prefix="T5_"):
    df = detail_df.copy()
    if "avg_unit_tech_cost" not in df.columns or "avg_unit_carbon_cost" not in df.columns:
        return False

    df[label_col] = df[label_col].astype(str).str.strip()
    df["avg_unit_tech_cost"] = pd.to_numeric(df["avg_unit_tech_cost"], errors="coerce")
    df["avg_unit_carbon_cost"] = pd.to_numeric(df["avg_unit_carbon_cost"], errors="coerce")
    df = df.dropna(subset=[label_col,"avg_unit_tech_cost","avg_unit_carbon_cost"]).copy()
    df = df.sort_values(label_col, key=lambda s: s.str.lower()).reset_index(drop=True)

    price_txt = f"${eq_price:.2f}/tCO₂" if eq_price is not None else "Pc N/A"
    fig, ax = plt.subplots(figsize=_figsize_for_categories(len(df), base_w=10, per_cat=0.22))

    x = np.arange(len(df))
    labels = df[label_col].astype(str).tolist()
    tech = df["avg_unit_tech_cost"].astype(float).values
    carb = df["avg_unit_carbon_cost"].astype(float).values
    total = tech + carb

    w = 0.38
    ax.bar(x - w/2, tech, width=w, label="Unit Tech Cost")
    ax.bar(x + w/2, carb, width=w, label="Unit Carbon Cost Change")
    ax.plot(x, total, marker="D", linewidth=1.5, label="Total (Tech + Carbon)")
    ax.axhline(0, linewidth=1.6)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_title(f"Unit Cost Decomposition by {level_name}\nScenario {scenario_id} at equilibrium Pc {price_txt}", fontweight="bold")
    ax.set_xlabel(level_name, fontweight="bold")
    ax.set_ylabel("Unit cost impact", fontweight="bold")
    ax.grid(True, axis="y", linestyle="--", alpha=0.3)
    ax.legend()
    fixed_save_plot(fig, outdir / f"{prefix}Fig1_Cost_Decomp_{level_name}_S{scenario_id}.png")
    return True

@fixed_plot_generation
def plot_scenario_comparison_heatmap(pivot_df: pd.DataFrame, title: str, outpath: Path, annotate=True):
    df = pivot_df.copy().replace([np.inf, -np.inf], np.nan).fillna(0)
    rows, cols = df.shape
    fig, ax = plt.subplots(figsize=(min(18, 4 + 1.2*cols), min(22, 3 + 0.30*rows)))
    im = ax.imshow(df.values, aspect="auto")
    ax.set_xticks(np.arange(cols))
    ax.set_xticklabels(df.columns.tolist(), fontweight="bold", rotation=45, ha="right")
    ax.set_yticks(np.arange(rows))
    ax.set_yticklabels(df.index.tolist(), fontsize=9, fontweight="bold")
    ax.set_title(title, fontweight="bold")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if annotate and rows <= 60 and cols <= 10:
        for i in range(rows):
            for j in range(cols):
                ax.text(j, i, f"{df.values[i,j]:.2f}", ha="center", va="center", fontsize=7, fontweight="bold")
    fixed_save_plot(fig, outpath)
    return True

# ============================================================
# STEP 6 UI
# ============================================================
step6_out = widgets.Output()
btn_launch_dashboard = widgets.Button(description="Launch Complete Dashboard 🚀", button_style="primary")

def launch_complete_dashboard(_=None):
    with step6_out:
        clear_output(wait=True)

        # Leak catcher: prevents callback output leaking to notebook cell
        leak_out = widgets.Output()
        leak_out.layout.display = "none"

        def _capture(fn):
            def wrapped(*args, **kwargs):
                with leak_out:
                    clear_output(wait=True)
                    return fn(*args, **kwargs)
            return wrapped

        DASH = {
            "path": None,
            "xls": None,
            "summary": None,
            "facility": None,
            "facility_sheet": None,
            "scenarios": [],
            "cache": {},
            "detail_cache": {}
        }

        default_file = str(Path(PATHS.get("scenario_results", "/content/Carbon_Market_Scenarios_Results_v3.xlsx")).resolve())

        # ---------- Widgets ----------
        path_w = widgets.Text(value=default_file, description="Results file:",
                              layout=widgets.Layout(width="850px"),
                              style={'description_width': '120px'})

        outdir_w = widgets.Text(value="dashboard_outputs", description="Output dir:",
                                layout=widgets.Layout(width="400px"))
        clear_w = widgets.Checkbox(value=True, description="Clear old PNGs")

        scen_w = widgets.Dropdown(description="Scenario:", disabled=True, layout=widgets.Layout(width="280px"))
        btn_prev = widgets.Button(description="◀ Prev", layout=widgets.Layout(width="90px"))
        btn_next = widgets.Button(description="Next ▶", layout=widgets.Layout(width="90px"))

        level_w = widgets.Dropdown(
            options=[("Aggregate (All facilities)","all"), ("Sector","sector"), ("Subsector","subsector")],
            value="sector", description="Level:", layout=widgets.Layout(width="320px")
        )

        dpi_w = widgets.Dropdown(options=[("Standard (120)", 120), ("High (300)", 300)],
                                 value=120, description="DPI:", layout=widgets.Layout(width="200px"))

        status_w = widgets.HTML(value="<div class='status-badge info'>Ready</div>")
        perf_monitor = widgets.HTML(
            "<div style='font-size:11px;color:#666;padding:8px;background:#f8f9fa;border-radius:4px;border:1px solid #e0e0e0;'>"
            "<strong>Performance:</strong> Cache: 0/0 (0%) | Load: — | Memory: 0MB"
            "</div>"
        )

        plots_sel = widgets.SelectMultiple(
            options=[
                ("Impacts (T1)", "impacts"),
                ("Bubble plots (B)", "bubble_pack"),
                ("Distributions (T2)", "dists"),
                ("Market mechanics (T3)", "market"),
                ("Buyer/Seller (T4)", "bs"),
                ("Cost decomposition (T5)", "costdecomp"),
                ("Scenario comparison (T6)", "compare"),
            ],
            value=("impacts","bubble_pack","dists"),
            description="Plots:",
            layout=widgets.Layout(width="900px", height="150px")
        )

        PRESETS = {
            "Core (recommended)": ("impacts","bubble_pack","dists"),
            "All plots": ("impacts","bubble_pack","dists","market","bs","costdecomp","compare"),
            "Only mechanics": ("market","bs"),
            "Only impacts": ("impacts",),
            "Only bubbles": ("bubble_pack",),
            "Only comparison": ("compare",),
        }
        preset_w = widgets.Dropdown(options=list(PRESETS.keys()), value="Core (recommended)",
                                    description="Preset:", layout=widgets.Layout(width="360px"))
        btn_apply_preset = widgets.Button(description="Apply preset", button_style="info",
                                          layout=widgets.Layout(width="140px"))
        btn_apply_preset.on_click(lambda _: setattr(plots_sel, "value", PRESETS[preset_w.value]))

        adv_toggle = widgets.Checkbox(value=False, description="Show advanced controls")
        clip_outliers = widgets.Checkbox(value=False, description="Clip outliers")
        neutral_band = widgets.FloatSlider(value=0.00, min=0.0, max=2.0, step=0.05,
                                           description="Neutral ±%:", readout_format=".2f",
                                           layout=widgets.Layout(width="520px"))
        sector_filter = widgets.SelectMultiple(options=[], value=(), description="Sector filter:",
                                               layout=widgets.Layout(width="520px", height="120px"))
        adv_box = widgets.VBox([clip_outliers, neutral_band, sector_filter])
        adv_box.layout.display = "none"
        adv_toggle.observe(lambda ch: setattr(adv_box.layout, "display", "" if ch["new"] else "none"), names="value")

        bubble_size_by_w = widgets.Dropdown(
            options=[("Bubble size: baseline production (q0)", "q0"),
                     ("Bubble size: post-ETS production (qst)", "qst")],
            value="q0", description="Bubble size:", layout=widgets.Layout(width="520px")
        )
        bubble_order_w = widgets.Dropdown(
            options=[("Sector order: alphabetical", "alpha"),
                     ("Sector order: median of x", "median")],
            value="alpha", description="Sector order:", layout=widgets.Layout(width="320px")
        )

        compare_scenarios = widgets.SelectMultiple(options=[], value=(), description="Compare scenarios:",
                                                   layout=widgets.Layout(width="520px", height="120px"))
        compare_metric = widgets.Dropdown(
            options=[
                ("Carbon intensity change (%)", "carbon_intensity_change_pct"),
                ("Unit cost ratio (% of price)", "unit_cost_change_ratio"),
                ("Output change (%)", "production_change_pct"),
                ("Avg unit total cost change", "avg_unit_total_cost_change"),
                ("Avg unit tech cost", "avg_unit_tech_cost"),
                ("Avg unit carbon cost", "avg_unit_carbon_cost"),
                ("Emission reduction (%)", "emission_reduction_pct"),
            ],
            value="carbon_intensity_change_pct",
            description="Metric:",
            layout=widgets.Layout(width="520px")
        )
        compare_annot = widgets.Checkbox(value=True, description="Annotate heatmap")

        prefix_filter = widgets.Dropdown(
            options=[
                ("All", ""),
                ("Impacts (T1_)", "T1_"),
                ("Distributions (T2_)", "T2_"),
                ("Market (T3_)", "T3_"),
                ("Buyer/Seller (T4_)", "T4_"),
                ("CostDecomp (T5_)", "T5_"),
                ("Compare (T6_)", "T6_"),
                ("Bubbles (B)", "B"),
            ],
            value="",
            description="Gallery:",
            layout=widgets.Layout(width="320px")
        )

        view_w = widgets.Dropdown(
            options=[("KPIs","kpi"), ("Plots","plots"), ("Compare","compare"), ("Gallery","gal")],
            value="kpi", description="View:", layout=widgets.Layout(width="260px")
        )

        # Outputs (dash_content avoids collision with your navigation wizard `content`)
        out_kpi = widgets.Output()
        out_plots = widgets.Output()
        out_compare = widgets.Output()
        out_gal = widgets.Output()
        dash_content = widgets.VBox([out_kpi])

        btn_refresh_gallery = widgets.Button(description="🔄 Refresh Gallery", button_style="info",
                                             layout=widgets.Layout(width="160px"))

        btn_load = widgets.Button(description="Load Data", button_style="info", layout=widgets.Layout(width="120px"))
        btn_clear_cache = widgets.Button(description="Clear Cache", button_style="warning", layout=widgets.Layout(width="120px"))
        btn_gen = widgets.Button(description="Generate Plots", button_style="success", disabled=True, layout=widgets.Layout(width="150px"))
        btn_zip = widgets.Button(description="Generate HD + Zip", button_style="warning", disabled=True, layout=widgets.Layout(width="180px"))
        btn_show = widgets.Button(description="Show Gallery", layout=widgets.Layout(width="120px"))
        btn_export = widgets.Button(description="Export Tables", layout=widgets.Layout(width="140px"))
        btn_clear_outputs = widgets.Button(description="Clear Outputs", button_style="warning", layout=widgets.Layout(width="140px"))

        gen_bar = widgets.IntProgress(value=0, min=0, max=100, description="Progress:",
                                      bar_style="info", layout=widgets.Layout(width="520px"))
        gen_msg = widgets.HTML("<div class='progress-label'>Ready.</div>")

        # ---------- Navigation buttons ----------
        def _move_scenario(step):
            opts = list(scen_w.options) if scen_w.options else []
            if not opts:
                return
            cur = opts.index(scen_w.value) if scen_w.value in opts else 0
            scen_w.value = opts[(cur + step) % len(opts)]

        btn_prev.on_click(lambda _: _move_scenario(-1))
        btn_next.on_click(lambda _: _move_scenario(+1))

        # ---------- Helpers ----------
        def cache_read(sheet_name: str):
            if sheet_name in DASH["cache"]:
                return DASH["cache"][sheet_name]
            df = optimized_read_excel(DASH["path"], sheet_name)
            DASH["cache"][sheet_name] = df
            return df

        def update_sector_options():
            if DASH["facility"] is None or DASH["facility"].empty:
                sector_filter.options = []
                sector_filter.value = ()
                return
            d = DASH["facility"][DASH["facility"]["scenario"].astype(str).str.strip() == str(scen_w.value).strip()].copy()
            if "sector" not in d.columns:
                sector_filter.options = []
                sector_filter.value = ()
                return
            sectors = sorted(d["sector"].astype(str).str.strip().dropna().unique().tolist(), key=lambda x: str(x).lower())
            sector_filter.options = sectors
            cur = set(sector_filter.value)
            sector_filter.value = tuple([s for s in sectors if s in cur])

        def _detail_cached(scen: str, level_choice: str, sec_filter_list):
            key = (str(scen).strip(), str(level_choice).strip(), tuple(sec_filter_list) if sec_filter_list else tuple())
            if key in DASH["detail_cache"]:
                return DASH["detail_cache"][key]
            detail_df, label_col, level_name = _get_detail_df(DASH["xls"], DASH["path"], DASH["facility"], scen, level_choice, cache_read)
            if sec_filter_list and label_col == "sector":
                detail_df = detail_df[detail_df["sector"].astype(str).isin(set(sec_filter_list))].copy()
            DASH["detail_cache"][key] = (detail_df, label_col, level_name)
            return detail_df, label_col, level_name

        def build_compare_pivot(metric: str, scenarios: list, level_choice: str):
            level_choice = str(level_choice).lower().strip()

            def _alpha_sort_index(df):
                return df.sort_index(key=lambda idx: idx.astype(str).str.lower())

            # all facilities
            if level_choice == "all":
                rows = []
                if DASH["facility"] is None:
                    if metric == "emission_reduction_pct" and DASH["summary"] is not None:
                        for sid in scenarios:
                            row = _get_summary_row(DASH["summary"], sid)
                            rows.append({"all": "All facilities", "Scenario": sid, metric: float(row.get("Emission_Reduction_Pct", 0.0))})
                        piv = pd.DataFrame(rows).pivot_table(index="all", columns="Scenario", values=metric, aggfunc="first").fillna(0)
                        piv = piv[[c for c in scenarios if c in piv.columns]]
                        return piv, "(all facilities; from Summary)"
                    raise ValueError("All facilities compare needs facility table (except emission reduction).")

                for sid in scenarios:
                    fsub = DASH["facility"][DASH["facility"]["scenario"].astype(str).str.strip() == str(sid).strip()].copy()
                    df_all = _detail_all_from_facility(fsub)
                    rows.append({"all": "All facilities", "Scenario": sid, metric: float(df_all.iloc[0].get(metric, 0.0))})
                piv = pd.DataFrame(rows).pivot_table(index="all", columns="Scenario", values=metric, aggfunc="first").fillna(0)
                piv = piv[[c for c in scenarios if c in piv.columns]]
                return piv, "(all facilities; computed)"

            # sector compare (computed)
            if level_choice == "sector":
                if DASH["facility"] is None:
                    raise ValueError("Sector compare requires facility table.")
                rows = []
                for sid in scenarios:
                    fsub = DASH["facility"][DASH["facility"]["scenario"].astype(str).str.strip() == str(sid).strip()].copy()
                    df_sec = _detail_from_facility(fsub, "sector")
                    if metric not in df_sec.columns:
                        raise ValueError(f"Metric '{metric}' missing for sector compare.")
                    tmp = df_sec[["sector", metric]].copy()
                    tmp["Scenario"] = sid
                    rows.append(tmp)
                long = pd.concat(rows, ignore_index=True)
                piv = long.pivot_table(index="sector", columns="Scenario", values=metric, aggfunc="first").fillna(0)
                piv = piv[[c for c in scenarios if c in piv.columns]]
                piv = _alpha_sort_index(piv)
                return piv, "(sector-level; computed)"

            # subsector pivot sheets
            pivot_map = {
                "carbon_intensity_change_pct": "Subsector_Intensity_Change",
                "unit_cost_change_ratio": "Subsector_Cost_Ratio_Pct",
                "production_change_pct": "Subsector_Production_Change",
                "avg_unit_total_cost_change": "Subsector_Avg_Total_Cost",
                "avg_unit_tech_cost": "Subsector_Avg_Tech_Cost",
                "avg_unit_carbon_cost": "Subsector_Avg_Carbon_Cost",
                "emission_reduction_pct": "Subsector_Emission_Red_Pct",
            }
            sheet = pivot_map.get(metric, None)
            if sheet and sheet in DASH["xls"].sheet_names:
                piv = cache_read(sheet)
                if "Subsector" in piv.columns:
                    piv = piv.set_index("Subsector")
                elif "Unnamed: 0" in piv.columns:
                    piv = piv.set_index("Unnamed: 0")
                else:
                    piv = piv.set_index(piv.columns[0])
                piv = piv.apply(pd.to_numeric, errors="coerce").fillna(0)
                piv = piv[[c for c in scenarios if c in piv.columns]]
                piv = _alpha_sort_index(piv)
                return piv, f"(subsector pivot sheet {sheet})"

            # subsector computed fallback
            if DASH["facility"] is None:
                raise ValueError("Subsector compare requires facility table or pivot sheet.")
            rows = []
            for sid in scenarios:
                fsub = DASH["facility"][DASH["facility"]["scenario"].astype(str).str.strip() == str(sid).strip()].copy()
                if "subsector" not in fsub.columns:
                    raise ValueError("Facility table missing 'subsector' column for subsector compare.")
                df_sub = _detail_from_facility(fsub, "subsector")
                if metric not in df_sub.columns:
                    raise ValueError(f"Metric '{metric}' missing for subsector compare.")
                tmp = df_sub[["subsector", metric]].copy()
                tmp["Scenario"] = sid
                rows.append(tmp)
            long = pd.concat(rows, ignore_index=True)
            piv = long.pivot_table(index="subsector", columns="Scenario", values=metric, aggfunc="first").fillna(0)
            piv = piv[[c for c in scenarios if c in piv.columns]]
            piv = _alpha_sort_index(piv)
            return piv, "(subsector; computed)"

        def render_kpis():
            with out_kpi:
                clear_output(wait=True)
                if DASH["summary"] is None or DASH["summary"].empty:
                    display(widgets.HTML("<div class='status-badge warning'>Summary not found.</div>"))
                    return

                sid = scen_w.value
                eqp = _get_eq_price(DASH["summary"], sid)
                row = _get_summary_row(DASH["summary"], sid)

                buyers = sellers = None
                if DASH["facility"] is not None and "N" in DASH["facility"].columns:
                    tmp = DASH["facility"][DASH["facility"]["scenario"].astype(str).str.strip()==str(sid).strip()].copy()
                    tmp["N"] = pd.to_numeric(tmp["N"], errors="coerce")
                    buyers = int((tmp["N"] > 0).sum())
                    sellers = int((tmp["N"] < 0).sum())

                def fmt(x, digits=2):
                    try: return f"{float(x):,.{digits}f}"
                    except: return "—"

                display(widgets.HTML(f"""
                <div style="display:flex; flex-wrap:wrap; gap:12px; margin-bottom:12px;">
                  <div style="border:1px solid #ddd; border-radius:10px; padding:10px; min-width:200px; background:#f8f9fa;">
                    <div style="font-size:12px;color:#666;">Scenario</div>
                    <div style="font-size:18px;font-weight:700;color:#2c3e50;">{sid}</div>
                  </div>
                  <div style="border:1px solid #ddd; border-radius:10px; padding:10px; min-width:240px; background:#f8f9fa;">
                    <div style="font-size:12px;color:#666;">Equilibrium Price</div>
                    <div style="font-size:18px;font-weight:700;color:#2c3e50;">${fmt(eqp,2)}/tCO₂</div>
                  </div>
                  <div style="border:1px solid #ddd; border-radius:10px; padding:10px; min-width:260px; background:#f8f9fa;">
                    <div style="font-size:12px;color:#666;">Emission reduction (%)</div>
                    <div style="font-size:18px;font-weight:700;color:#2c3e50;">{fmt(row.get('Emission_Reduction_Pct','—'),2)}%</div>
                  </div>
                  <div style="border:1px solid #ddd; border-radius:10px; padding:10px; min-width:240px; background:#f8f9fa;">
                    <div style="font-size:12px;color:#666;">Buyers / Sellers</div>
                    <div style="font-size:18px;font-weight:700;color:#2c3e50;">{buyers if buyers is not None else '—'} / {sellers if sellers is not None else '—'}</div>
                  </div>
                </div>
                """))
                try:
                    display(pd.DataFrame([row]))
                except Exception:
                    pass

        # ✅ SAFE gallery: show last N only; no auto dump after generation
        def on_refresh_gallery(_=None):
            with out_gal:
                clear_output(wait=True)
                out_dir = Path(outdir_w.value)
                out_dir.mkdir(parents=True, exist_ok=True)

                pref = prefix_filter.value if prefix_filter.value else None
                pngs = sorted(out_dir.glob("*.png"), key=lambda p: p.stat().st_mtime)

                if pref == "B":
                    pngs = [p for p in pngs if p.name.startswith("B")]
                elif pref:
                    pngs = [p for p in pngs if p.name.startswith(pref)]

                total = len(pngs)
                if total == 0:
                    display(widgets.HTML("<div class='status-badge info'>No PNGs found (generate plots / check filters).</div>"))
                    return

                show = pngs[-MAX_GALLERY_SHOW:]
                display(widgets.HTML(
                    f"<div class='status-badge success'>Gallery: showing last {len(show)} of {total} PNGs in <code>{out_dir}</code></div>"
                ))

                for i, p in enumerate(show, start=max(1, total - len(show) + 1)):
                    try:
                        sz_kb = p.stat().st_size / 1024
                        display(widgets.HTML(f"<div class='plot-title'>{i}. {p.name} ({sz_kb:.1f} KB)</div>"))
                        img = Image(filename=str(p))
                        img.width = 900 if ("heatmap" in p.name.lower() or "T6_" in p.name) else 800
                        display(img)
                        display(widgets.HTML("<hr style='margin: 16px 0; border: none; border-top: 2px dashed #ddd;'>"))
                    except Exception as e:
                        display(widgets.HTML(f"<div class='status-badge error'>Error displaying {p.name}: {e}</div>"))

        def set_view(_=None):
            if view_w.value == "kpi":
                dash_content.children = [out_kpi]
                render_kpis()
            elif view_w.value == "plots":
                dash_content.children = [out_plots]
            elif view_w.value == "compare":
                dash_content.children = [out_compare]
            elif view_w.value == "gal":
                dash_content.children = [out_gal]
                with out_gal:
                    clear_output(wait=True)
                    display(widgets.HTML(
                        f"<div class='status-badge info'>Gallery ready. Click <b>Refresh Gallery</b> (shows last {MAX_GALLERY_SHOW}).</div>"
                    ))

        view_w.observe(set_view, names="value")

        # ---------- Actions ----------
        def enhanced_on_load(_=None):
            global PLOT_DPI
            PLOT_DPI = int(dpi_w.value)

            btn_load.disabled = True
            btn_gen.disabled = True
            btn_zip.disabled = True

            start = time.time()
            status_w.value = "<div class='status-badge info'>Loading...</div>"
            gen_bar.value = 10

            p = Path(path_w.value)
            if not p.exists():
                status_w.value = "<div class='status-badge error'>File not found</div>"
                gen_bar.value = 0
                btn_load.disabled = False
                return

            try:
                ENHANCED_CACHE.clear()
                DASH["cache"].clear()
                DASH["detail_cache"].clear()

                gen_bar.value = 30
                DASH["path"] = p.resolve()
                DASH["xls"] = _open_excel(DASH["path"])

                gen_bar.value = 55
                DASH["summary"] = optimized_read_excel(DASH["path"], "Summary") if "Summary" in DASH["xls"].sheet_names else None

                gen_bar.value = 75
                fac_sheet = _find_facility_sheet(DASH["xls"].sheet_names)
                DASH["facility_sheet"] = fac_sheet
                DASH["facility"] = optimized_read_excel(DASH["path"], fac_sheet) if fac_sheet else None

                gen_bar.value = 90
                DASH["scenarios"] = _available_scenarios_from_excel(DASH["xls"], DASH["path"])

                if not DASH["scenarios"]:
                    status_w.value = "<div class='status-badge warning'>No scenarios found</div>"
                    gen_bar.value = 0
                    return

                scen_w.options = DASH["scenarios"]
                scen_w.value = "B" if "B" in DASH["scenarios"] else DASH["scenarios"][0]
                scen_w.disabled = False

                compare_scenarios.options = DASH["scenarios"]
                compare_scenarios.value = tuple(DASH["scenarios"])

                update_sector_options()

                btn_gen.disabled = False
                btn_zip.disabled = False

                cache_stats = ENHANCED_CACHE.stats()
                load_time = time.time() - start
                perf_monitor.value = (
                    "<div style='font-size: 11px; color: #666; padding: 8px; background: #f8f9fa; border-radius: 4px; border: 1px solid #e0e0e0;'>"
                    f"<strong>Performance:</strong> Cache hit rate: {cache_stats['hit_rate']} | "
                    f"Load: {load_time:.1f}s | Memory: {cache_stats['size_mb']:.1f}MB"
                    "</div>"
                )

                status_w.value = f"<div class='status-badge success'>Loaded {len(DASH['scenarios'])} scenarios</div>"
                render_kpis()

                set_view()
                gen_bar.value = 100

            except Exception as e:
                status_w.value = f"<div class='status-badge error'>Error: {str(e)[:120]}</div>"
                with out_compare:
                    clear_output(wait=True)
                    print("Detailed error:", e)
                gen_bar.value = 0
            finally:
                gen_bar.value = 0
                btn_load.disabled = False

        def enhanced_generate_plots(_=None):
            global PLOT_DPI
            PLOT_DPI = int(dpi_w.value)

            if DASH["xls"] is None:
                status_w.value = "<div class='status-badge warning'>Load data first</div>"
                return

            btn_gen.disabled = True
            btn_zip.disabled = True
            btn_load.disabled = True

            status_w.value = "<div class='status-badge info'>Generating plots...</div>"
            gen_bar.value = 10
            start = time.time()

            scen = scen_w.value
            out_dir = Path(outdir_w.value)
            out_dir.mkdir(parents=True, exist_ok=True)

            if clear_w.value:
                _clear_pngs(out_dir)

            try:
                gen_bar.value = 25
                eqp = _get_eq_price(DASH["summary"], scen) if DASH["summary"] is not None else None
                sec_filter = list(sector_filter.value) if sector_filter.value else None

                detail_df, label_col, level_name = (None, None, None)
                try:
                    detail_df, label_col, level_name = _detail_cached(scen, level_w.value, sec_filter)
                except Exception:
                    detail_df = None

                selected_plots = list(plots_sel.value)

                gen_bar.value = 40
                if "impacts" in selected_plots and detail_df is not None:
                    plot_impacts_by_level(detail_df, label_col, level_name, scen, eqp, out_dir, prefix="T1_")

                gen_bar.value = 55
                if "dists" in selected_plots and DASH["facility"] is not None:
                    plot_equilibrium_distributions(
                        DASH["facility"], scen, eqp, out_dir,
                        neutral_band_pct=float(neutral_band.value),
                        clip_outliers=bool(clip_outliers.value),
                        sector_filter=sec_filter,
                        prefix="T2_"
                    )

                gen_bar.value = 65
                if "bubble_pack" in selected_plots and DASH["facility"] is not None:
                    plot_bubble_pack(
                        DASH["facility"], scen, eqp, out_dir,
                        size_by=bubble_size_by_w.value,
                        order_mode=bubble_order_w.value,
                        neutral_band_pct=float(neutral_band.value),
                        include_counts_plot=False,
                        sector_filter=sec_filter
                    )

                gen_bar.value = 75
                if "market" in selected_plots:
                    plot_market_mechanics_price_on_y(cache_read, DASH["xls"], scen, eqp, out_dir, prefix="T3_")

                gen_bar.value = 82
                if "bs" in selected_plots and DASH["facility"] is not None:
                    plot_buyer_seller_structure(DASH["facility"], scen, eqp, out_dir, sector_filter=sec_filter, prefix="T4_")

                gen_bar.value = 88
                if "costdecomp" in selected_plots and detail_df is not None:
                    plot_cost_decomposition(detail_df, label_col, level_name, scen, eqp, out_dir, prefix="T5_")

                gen_bar.value = 92
                if "compare" in selected_plots:
                    scens = list(compare_scenarios.value) if compare_scenarios.value else list(DASH["scenarios"])
                    piv, note = build_compare_pivot(compare_metric.value, scens, level_w.value)
                    plot_scenario_comparison_heatmap(
                        piv,
                        title=f"Scenario comparison ({level_w.value}) — {compare_metric.value} {note}",
                        outpath=out_dir / f"T6_Heatmap_{compare_metric.value}_{level_w.value}.png",
                        annotate=bool(compare_annot.value)
                    )
                    with out_compare:
                        clear_output(wait=True)
                        display(piv.head(20))

                gen_bar.value = 98
                n_png = len(list(out_dir.glob("*.png")))
                elapsed = time.time() - start
                status_w.value = f"<div class='status-badge success'>✅ Generated {n_png} PNG(s) in {elapsed:.1f}s</div>"

                # ✅ Do NOT auto-render gallery (prevents crash). Provide prompt instead.
                with out_gal:
                    clear_output(wait=True)
                    display(widgets.HTML(
                        f"<div class='status-badge success'>Done ✅ Generated {n_png} PNG(s). Click <b>Refresh Gallery</b> (shows last {MAX_GALLERY_SHOW}).</div>"
                    ))
                gen_bar.value = 100

            except Exception as e:
                status_w.value = f"<div class='status-badge error'>❌ Error: {str(e)[:120]}</div>"
                with out_gal:
                    clear_output(wait=True)
                    display(widgets.HTML(f"<div class='status-badge error'>Plot generation failed: {e}</div>"))
                gen_bar.value = 0
            finally:
                gen_bar.value = 0
                btn_gen.disabled = False
                btn_zip.disabled = False
                btn_load.disabled = False
                plt.close("all")
                gc.collect()

        def on_zip(_=None):
            old = dpi_w.value
            dpi_w.value = 300
            try:
                enhanced_generate_plots(None)
                out_dir = Path(outdir_w.value)
                zip_base = str(out_dir.resolve())
                zip_path = shutil.make_archive(zip_base, "zip", root_dir=str(out_dir.resolve()))
                status_w.value = f"<div class='status-badge success'>Zip created: {Path(zip_path).name}</div>"
                try:
                    from google.colab import files
                    files.download(zip_path)
                except Exception:
                    pass
            finally:
                dpi_w.value = old

        def on_export(_=None):
            if DASH["xls"] is None:
                status_w.value = "<div class='status-badge warning'>Load data first</div>"
                return

            with out_compare:
                clear_output(wait=True)
                scen = scen_w.value
                out_dir = Path(outdir_w.value)
                out_dir.mkdir(parents=True, exist_ok=True)
                export_path = out_dir / f"Dashboard_Tables_{scen}_{level_w.value}.xlsx"

                srow = _get_summary_row(DASH["summary"], scen) if DASH["summary"] is not None else {}
                try:
                    detail_df, label_col, level_name = _detail_cached(scen, level_w.value, list(sector_filter.value) if sector_filter.value else None)
                except Exception:
                    detail_df = pd.DataFrame()

                facility_sub = pd.DataFrame()
                if DASH["facility"] is not None:
                    facility_sub = DASH["facility"][DASH["facility"]["scenario"].astype(str).str.strip()==str(scen).strip()].copy()

                with pd.ExcelWriter(export_path, engine="openpyxl") as w:
                    pd.DataFrame([srow]).to_excel(w, sheet_name="Summary_Row", index=False)
                    detail_df.to_excel(w, sheet_name="Detail", index=False)
                    facility_sub.to_excel(w, sheet_name="Facility_Subset", index=False)

                print("Exported:", export_path.resolve())
                try:
                    from google.colab import files
                    files.download(str(export_path.resolve()))
                except Exception:
                    pass

        def on_clear_cache(_=None):
            ENHANCED_CACHE.clear()
            DASH["cache"].clear()
            DASH["detail_cache"].clear()
            gc.collect()
            status_w.value = "<div class='status-badge success'>Cache cleared</div>"
            perf_monitor.value = (
                "<div style='font-size:11px;color:#666;padding:8px;background:#f8f9fa;border-radius:4px;border:1px solid #e0e0e0;'>"
                "<strong>Performance:</strong> Cache: 0/0 (0%) | Load: — | Memory: 0MB"
                "</div>"
            )

        def on_clear_outputs(_=None):
            with out_kpi: clear_output(wait=True)
            with out_plots: clear_output(wait=True)
            with out_compare: clear_output(wait=True)
            with out_gal: clear_output(wait=True)
            status_w.value = "<div class='status-badge info'>Outputs cleared</div>"

        def on_show_gallery(_=None):
            view_w.value = "gal"
            set_view()

        def on_scenario_change(_=None):
            if DASH["xls"] is not None:
                update_sector_options()
                render_kpis()

        # ---------- Bind handlers ----------
        scen_w.observe(on_scenario_change, names="value")
        btn_refresh_gallery.on_click(_capture(on_refresh_gallery))
        btn_load.on_click(_capture(enhanced_on_load))
        btn_clear_cache.on_click(_capture(on_clear_cache))
        btn_clear_outputs.on_click(_capture(on_clear_outputs))
        btn_gen.on_click(_capture(enhanced_generate_plots))
        btn_zip.on_click(_capture(on_zip))
        btn_show.on_click(_capture(on_show_gallery))
        btn_export.on_click(_capture(on_export))

        # ---------- UI ----------
        header = widgets.HTML("""
        <div class="dashboard-title">
            <h2 style="margin: 0; color: #2c3e50;">📊 Carbon Market Dashboard</h2>
            <p style="margin: 5px 0 0 0; color: #666; font-size: 14px;"></p>
        </div>
        """)

        dashboard = widgets.VBox([
            header,
            status_w,
            perf_monitor,

            widgets.HTML("<b>📁 Data Source</b>"),
            widgets.HBox([path_w, btn_load, btn_clear_cache]),
            widgets.HBox([outdir_w, clear_w, btn_clear_outputs]),

            widgets.HTML("<b>🎯 Scenario</b>"),
            widgets.HBox([scen_w, btn_prev, btn_next, level_w, dpi_w],
                         layout=widgets.Layout(flex_flow="row wrap", gap="10px")),

            widgets.HTML("<b>📈 Plot selection</b>"),
            widgets.HBox([preset_w, btn_apply_preset]),
            plots_sel,

            widgets.HTML("<b>⚙️ Advanced</b>"),
            widgets.HBox([adv_toggle]),
            adv_box,
            widgets.HTML("<b>Bubble settings</b>"),
            widgets.HBox([bubble_size_by_w, bubble_order_w],
                         layout=widgets.Layout(flex_flow="row wrap", gap="10px")),

            widgets.HTML("<b>📊 Comparison</b>"),
            widgets.HBox([compare_scenarios, compare_metric, compare_annot],
                         layout=widgets.Layout(flex_flow="row wrap", gap="10px")),

            widgets.HTML("<b>🖼️ Gallery</b>"),
            widgets.HBox([view_w, prefix_filter, btn_show, btn_refresh_gallery],
                         layout=widgets.Layout(flex_flow="row wrap", gap="10px")),

            gen_bar,
            gen_msg,

            widgets.HBox([btn_gen, btn_zip, btn_export],
                         layout=widgets.Layout(justify_content="center", gap="15px")),

            dash_content
        ], layout=widgets.Layout(padding="20px", width="100%"))

        display(dashboard)

        # Auto-load after UI mounts (Colab quirk)
        if Path(default_file).exists():
            try:
                btn_load.click()
            except Exception:
                enhanced_on_load(None)

# Prevent duplicate click handlers if this cell is rerun
try:
    btn_launch_dashboard._click_handlers.callbacks = []
except Exception:
    pass
btn_launch_dashboard.on_click(launch_complete_dashboard)

# Step 6 container used by your navigation wizard
step6_box = widgets.VBox([
    widgets.HTML("<h3>Step 6 — Complete Results Dashboard</h3>"),
    btn_launch_dashboard,
    step6_out
])


HTML(value='\n<style>\n.widget-label { white-space: normal !important; }\n.widget-html { white-space: normal !…

# step 7

In [ ]:
# ======================================================
# STEP 7 — FINAL NAVIGATION WIZARD
# ======================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

# Ensure registry exists (steps can register themselves)
WIZARD_STEPS = globals().setdefault("WIZARD_STEPS", {})

# Map wizard keys -> fallback global variable names (legacy support)
STEP_MAPPING = {
    "s1": "step1_box",
    "s2": "step2_box",
    "s3": "step3_box",
    "s4": "step4_box",
    "s5": "step5_box",
    "s6": "step6_box",
}

# Controls
nav = widgets.Dropdown(
    options=[
        ("Step 1 — 📂 File Status", "s1"),
        ("Step 2 — 🛠️ Filter Reference", "s2"),
        ("Step 3 — ⚖️ Scale MACCs", "s3"),
        ("Step 4 — 🧩 Build Merged Panel", "s4"),
        ("Step 5 — 🚀 Scenario Simulation", "s5"),
        ("Step 6 — 📊 Results Dashboard", "s6"),
    ],
    value="s1",
    description="Select Step:",
    layout=widgets.Layout(width="360px"),
)

btn_go = widgets.Button(
    description="Go",
    button_style="primary",
    icon="arrow-right",
    layout=widgets.Layout(width="90px"),
)

btn_refresh = widgets.Button(
    description="Refresh",
    button_style="info",
    icon="refresh",
    tooltip="Reload registry/globals and re-render the current step",
    layout=widgets.Layout(width="110px"),
)

btn_diag = widgets.Button(
    description="Diagnostics",
    button_style="",
    icon="search",
    tooltip="Show which step widgets are currently available",
    layout=widgets.Layout(width="130px"),
)

content_out = widgets.Output()

# ---------- Resolver ----------
def _resolve_step_widget(step_key: str):
    """
    Resolution order:
      1) WIZARD_STEPS[step_key] (recommended, most robust)
      2) globals()[STEP_MAPPING[step_key]] (fallback)
    Returns: (widget_or_None, source_label, missing_var_name_or_None)
    """
    if step_key in WIZARD_STEPS:
        return WIZARD_STEPS[step_key], f"WIZARD_STEPS['{step_key}']", None

    var_name = STEP_MAPPING.get(step_key)
    if var_name and (var_name in globals()):
        return globals()[var_name], f"globals()['{var_name}']", None

    return None, None, var_name

# ---------- Renderer ----------
def navigate_to_step(_=None):
    step_key = nav.value
    widget_obj, source, missing_name = _resolve_step_widget(step_key)

    with content_out:
        clear_output(wait=True)

        if widget_obj is not None:
            # Try display normally; if something fails, show error box
            try:
                display(widget_obj)
            except Exception as e:
                display(widgets.HTML(
                    "<div style='padding:16px; border:2px solid #e74c3c; border-radius:8px; background:#fdecea;'>"
                    f"<h3 style='color:#c0392b; margin:0 0 8px 0;'>⚠️ Display failed ({source})</h3>"
                    f"<pre style='white-space:pre-wrap; margin:0;'>{e}</pre>"
                    "</div>"
                ))
        else:
            step_num = step_key[-1]
            display(widgets.HTML(
                "<div style='padding:18px; border:2px solid #e74c3c; border-radius:8px; background:#fdecea;'>"
                f"<h3 style='color:#c0392b; margin-top:0;'>⚠️ Missing Step {step_num}</h3>"
                f"<p>Could not find <b>{missing_name}</b> in <code>globals()</code> and no registry entry in "
                f"<code>WIZARD_STEPS['{step_key}']</code>.</p>"
                "<b>How to fix:</b>"
                "<ol style='margin:8px 0 0 18px;'>"
                f"<li>Run the notebook cell that defines <code>{missing_name}</code>.</li>"
                "<li>At the end of that step cell, add registry lines (recommended):<br>"
                f"<code>globals()['{missing_name}'] = {missing_name}<br>"
                f"globals().setdefault('WIZARD_STEPS', {{}})['{step_key}'] = {missing_name}</code></li>"
                "</ol>"
                "</div>"
            ))

# ---------- Diagnostics ----------
def show_diagnostics(_=None):
    with content_out:
        clear_output(wait=True)

        # Things that look like step boxes in globals
        global_boxes = sorted([k for k in globals().keys() if k.startswith("step") and k.endswith("_box")])
        registry_keys = sorted(WIZARD_STEPS.keys())

        # Build a status table
        rows = []
        for k, var in STEP_MAPPING.items():
            reg_ok = k in WIZARD_STEPS
            glob_ok = var in globals()
            rows.append((k, var, "✅" if reg_ok else "—", "✅" if glob_ok else "—"))

        df_html = (
            "<table style='border-collapse:collapse; width:100%; font-size:13px;'>"
            "<tr>"
            "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px;'>Step key</th>"
            "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px;'>Global name</th>"
            "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px;'>Registry</th>"
            "<th style='text-align:left; border-bottom:1px solid #ddd; padding:6px;'>Globals</th>"
            "</tr>"
        )
        for a, b, c, d in rows:
            df_html += (
                "<tr>"
                f"<td style='border-bottom:1px solid #eee; padding:6px;'><code>{a}</code></td>"
                f"<td style='border-bottom:1px solid #eee; padding:6px;'><code>{b}</code></td>"
                f"<td style='border-bottom:1px solid #eee; padding:6px;'>{c}</td>"
                f"<td style='border-bottom:1px solid #eee; padding:6px;'>{d}</td>"
                "</tr>"
            )
        df_html += "</table>"

        display(widgets.HTML(
            "<div style='padding:14px; border:1px solid #ddd; border-radius:8px; background:#fafafa;'>"
            "<h3 style='margin:0 0 10px 0;'>Diagnostics</h3>"
            "<div style='color:#555; font-size:12px; margin-bottom:10px;'>"
            "Registry is preferred (WIZARD_STEPS). If a step is missing, run that step cell and ensure it registers."
            "</div>"
            f"{df_html}"
            "<hr style='margin:12px 0;'>"
            f"<div><b>Globals step boxes found:</b> <code>{global_boxes}</code></div>"
            f"<div><b>Registry keys found:</b> <code>{registry_keys}</code></div>"
            "</div>"
        ))

# ---------- Refresh ----------
def refresh_current(_=None):
    # Pull latest reference to registry (in case other cells overwrote it)
    globals()["WIZARD_STEPS"] = globals().setdefault("WIZARD_STEPS", {})
    navigate_to_step()

btn_go.on_click(navigate_to_step)
btn_refresh.on_click(refresh_current)
btn_diag.on_click(show_diagnostics)

# ---------- Layout ----------
title = widgets.HTML("<h2 style='margin:0 0 6px 0;'>🏭 Carbon Market Simulation Tool</h2>"
                     "<div style='color:#666; font-size:12px; margin-bottom:10px;'>"
                     "Use the dropdown to navigate steps. If a step is missing, run that step’s cell first."
                     "</div>")

controls = widgets.HBox(
    [nav, btn_go, btn_refresh, btn_diag],
    layout=widgets.Layout(
        background_color="#f8f9fa",
        padding="14px",
        border="1px solid #ddd",
        margin="0 0 12px 0",
        gap="10px",
        width="100%"
    )
)

# Display UI first (important), then render selected step
display(title)
display(controls)
display(content_out)

# Initial render
navigate_to_step()


HTML(value="<h2 style='margin:0 0 6px 0;'>🏭 Carbon Market Simulation Tool</h2><div style='color:#666; font-siz…

Output()

# dash